# GeoNUSAF — all figures, one notebook (v2)

Everything from `geonusaf_figures.ipynb` (plots) and `geonusaf_figures_diagrams.ipynb`
(block diagrams + dataset montages), with a group selector.

**How to use**

1. Set `CHOICE` and anything else you want in the **control panel** (the next cell).
2. Run All. Cells 1–12 only define things; the build happens in the last cell.

| `CHOICE` | builds |
|---|---|
| `"1"` | the 16 plotted figures |
| `"2"` | the 5 block diagrams |
| `"3"` | the 6 dataset photo montages (+ graphical abstract) |
| `"1,3"` | any combination |
| `"all"` | all 27 |

Each run writes into `figs/`, updates one `figs/figure_register.csv`, and rezips
`geonusaf_figures_all.zip` in `/kaggle/working` with a download link. `run("2")` and
`build("fig12_downstream_forest_plot")` work at any time after the setup cells.

**What changed from v1, and only this**

- **Section 5 finds your dataset itself.** It scans `/kaggle/input`, works out which
  folders hold images and which hold masks, and — importantly — *measures* the colour
  encoding of your masks rather than assuming one. The DEMO tiles you saw were the
  fallback firing because the hard-coded paths did not match your mount.
- **Section 8 picks the tiles by counting pixels**, and prints the reason for every
  choice. Anything you set by hand in the control panel wins.
- **Fig. 1a is now a drawn locator map** (Natural Earth outline, valley extent, scale
  bar). The "paste the QGIS export here" text is gone.
- **Fig. 2c computes all fifteen Jeffries–Matusita pairs from the corpus**, so the `?`
  cells and the INCOMPLETE note disappear on their own.
- Sections 1–4 and groups 1 and 2 are otherwise **untouched**: same rcParams, same
  Times stack, same 90 / 140 / 190 mm widths, same `finish()`, same figure code.


## 0 · Control panel — set this, then Run All

In [ ]:
# =====================================================================================
#   CONTROL PANEL  -  the only cell you normally touch.  Set it, then Run All.
#   Everything below this cell only defines things; the build happens in the last cell.
# =====================================================================================

CHOICE = "all"        # "1" plots · "2" block diagrams · "3" dataset photos · "1,3" · "all"

# ---- where your data is.  "auto" scans /kaggle/input and finds the folders itself,
#      whatever they are called and however deeply they are nested.
IMG_DIR   = "auto"
MASK_DIR  = "auto"
FAKE_IMG  = "auto"
FAKE_MASK = "auto"
PRED_DIR  = ""        # a folder of prediction PNGs, if you already have them

# ---- manual picks.  Anything you set here wins.  Anything left empty is chosen for
#      you from the pixel census, and the reason for each choice is printed.
FIG1_TILES = []                          # e.g. ["image120", "image264", "image301"]
FIG6_TILE, FIG6_CROP = "", None          # crop as (x0, y0, x1, y1) in full-tile pixels
FIG9_SCENES, FIG9_PRED_B0, FIG9_PRED_TC = [], [], []
FIG11_PAIRS, FIG11_READBACK = [], []
FIGA1_TILE, FIGA1_CROP = "", None
LOCATOR_IMAGE = ""                       # your own QGIS / satellite export for Fig. 1a;
                                         # empty = the drawn locator map is used

AUTO_PICK = True
USE_DISPLAY_PALETTE = True               # False renders the released mask colours instead
SEED = 42

# ---- how index-coded masks map to classes.  Only used for masks stored as 0..5 rather
#      than as colours - which is how the Stage-3 generated masks are stored.  This is
#      the class order in your model cards.
CLASS_ORDER_IN_MASKS = ["Residential", "Road", "River", "Forest", "UnusedLand", "Agricultural"]

# ---- Fig. 2c.  "auto" computes all fifteen Jeffries-Matusita pairs from the corpus and
#      checks the five recorded values against the computed ones before using them.
#      False keeps the five recorded values and the "?" cells.
JM_FROM_DATA = "auto"
JM_MAX_TILES = 150            # tiles sampled for the statistics; all of them if fewer

# ---- Fig. 9 and Fig. 11 need model output.  "auto" runs your checkpoints when this
#      session has internet; "off" never touches the network.  Nothing is ever invented:
#      if a model cannot be loaded the panel says so instead of showing something wrong.
INFER = "auto"
HF_B0 = "Pranilllllll/segformer-b0-kaggle-ktm-dataset-block-fold1"
HF_TC = "sugam24/geonusaf-tcsegformer-block-fold1"
FOLD, SPLIT_MODE = 1, "block"
MAX_SCORE_SCENES = 60         # how many validation scenes to score when ranking Fig. 9
BUILD_TC_MODEL = None         # paste a callable returning your TC-SegFormer module here
                              # if the built-in reconstruction does not match your keys

print("control panel set · CHOICE =", CHOICE)

## 1 · Setup, fonts, and house style

Unchanged.

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, TwoSlopeNorm
from matplotlib.patches import Patch, FancyBboxPatch, FancyArrowPatch, Circle, Rectangle
from matplotlib.lines import Line2D
import matplotlib.font_manager as fm

OUT = Path("figs"); OUT.mkdir(exist_ok=True)
SAVE_EPS = False     # Elsevier accepts EPS or PDF; PDF is the safer of the two
SAVE_PNG = True      # 600 dpi previews for checking. Submit the PDFs, not these.

MM = 1 / 25.4
W1, W15, W2 = 90 * MM, 140 * MM, 190 * MM     # Elsevier: single / 1.5 / double column

# ---- Times. Liberation Serif and Nimbus Roman are metrically identical to Times New Roman,
# ---- so on Kaggle or Linux the output is indistinguishable in print.
SERIF = ["Times New Roman", "Nimbus Roman", "Liberation Serif", "TeX Gyre Termes",
         "FreeSerif", "DejaVu Serif"]
have = {f.name for f in fm.fontManager.ttflist}
resolved = next((f for f in SERIF if f in have), "DejaVu Serif")
print("serif family in use:", resolved)
if resolved == "DejaVu Serif":
    print("  Times not found. On Kaggle run:  !apt-get -qq install -y fonts-liberation")
    print("  then restart the kernel. DejaVu Serif is a safe stand-in meanwhile.")

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 600,
    "savefig.facecolor": "white",
    "savefig.transparent": False,
    "pdf.fonttype": 42,          # embed TrueType. Type 3 fails Elsevier preflight.
    "ps.fonttype": 42,
    "font.family": "serif",
    "font.serif": SERIF,
    "mathtext.fontset": "stix",  # Times-compatible math
    "font.size": 7,              # Elsevier floor is 7 pt at final size
    "axes.labelsize": 7.5,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "legend.frameon": False,
    "legend.handlelength": 1.3,
    "legend.handletextpad": 0.5,
    "legend.columnspacing": 1.4,
    "legend.borderpad": 0.0,
    "axes.titlesize": 7.5,
    "axes.linewidth": 0.6,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "grid.linewidth": 0.4,
    "grid.color": "0.88",
    "lines.linewidth": 1.0,
    "lines.markersize": 3.5,
    "xtick.major.width": 0.6,
    "ytick.major.width": 0.6,
    "xtick.major.size": 2.4,
    "ytick.major.size": 2.4,
    "hatch.linewidth": 0.5,
})

SUBLABEL_PT = 8      # size of (a) (b) (c)
NOTE = dict(fontsize=5.6, color="0.55", style="italic")   # rare in-figure note style

REGISTER = []

def top_legend(fig, handles, labels, ncol, loc="outside upper center"):
    # Legend above the plot, outside the axes, so it can never sit on the data.
    return fig.legend(handles=handles, labels=labels, loc=loc, ncol=ncol,
                      frameon=False, fontsize=7, borderaxespad=0.15)

def finish(fig, name, figure_no, section, where, float_env, caption, axes=None, hspace=0.12):
    # axes: pass the panel axes in order to get (a) (b) (c) below them, on one baseline.
    # An entry may itself be a list of axes that share one sub-label (Fig. 1b).
    if axes:
        h_in = fig.get_size_inches()[1]
        reserve = (5.6 * MM) / h_in            # strip of figure height kept free for the labels
        eng = fig.get_layout_engine()

        from matplotlib.transforms import Bbox

        def boxes_now():
            fig.draw_without_rendering()
            r = fig.canvas.get_renderer()
            out = []
            for item in axes:
                group_ = item if isinstance(item, (list, tuple)) else [item]
                bs = [a.get_tightbbox(r).transformed(fig.transFigure.inverted())
                      for a in group_]
                out.append(Bbox.from_extents(min(b.x0 for b in bs), min(b.y0 for b in bs),
                                             max(b.x1 for b in bs), max(b.y1 for b in bs)))
            return out

        eng.set(rect=(0, reserve, 1, 1 - reserve))   # (left, bottom, width, height)
        bx = boxes_now()

        def cluster(bs):
            # group panels into rows by vertical centre, top row first
            idx = sorted(range(len(bs)), key=lambda i: -0.5 * (bs[i].y0 + bs[i].y1))
            rows, cur = [], [idx[0]]
            for i in idx[1:]:
                c0 = 0.5 * (bs[cur[-1]].y0 + bs[cur[-1]].y1)
                c1 = 0.5 * (bs[i].y0 + bs[i].y1)
                if abs(c0 - c1) < 0.06:
                    cur.append(i)
                else:
                    rows.append(cur); cur = [i]
            rows.append(cur)
            return rows

        rows = cluster(bx)
        if len(rows) > 1:            # stacked panels need a gap to hold the label
            eng.set(hspace=hspace)
            bx = boxes_now(); rows = cluster(bx)
        for row in rows:
            ybase = min(bx[i].y0 for i in row) - 0.30 * reserve
            for i in row:
                fig.text(0.5 * (bx[i].x0 + bx[i].x1), ybase, "(" + "abcdefgh"[i] + ")",
                         ha="center", va="top", fontsize=SUBLABEL_PT)
        fig.set_layout_engine("none")

    fig.savefig(OUT / (name + ".pdf"))
    if SAVE_EPS:
        fig.savefig(OUT / (name + ".eps"))
    if SAVE_PNG:
        fig.savefig(OUT / (name + ".png"))
    w, h = fig.get_size_inches()
    size = str(round(w / MM)) + " x " + str(round(h / MM)) + " mm"
    print(name + ".pdf   " + size + "   " + float_env)
    print("   " + figure_no + "  ->  " + section + ", " + where)
    REGISTER.append(dict(file=name + ".pdf", figure=figure_no, section=section,
                         insertion_point=where, float_env=float_env, size=size,
                         caption=caption))

print("ready. output ->", OUT.resolve())

## 2 · Palette

In [ ]:
# USE_DISPLAY_PALETTE comes from the control panel at the top.
CLASSES = ["Residential", "Agricultural", "UnusedLand", "Forest", "Road", "River"]

RELEASED = {"Residential": "#000080", "Agricultural": "#800080", "UnusedLand": "#008080",
            "Forest": "#808000", "Road": "#008000", "River": "#800000",
            "Unannotated": "#000000"}
DISPLAY  = {"Residential": "#C0392B", "Agricultural": "#E8B62C", "UnusedLand": "#C2A477",
            "Forest": "#14532D", "Road": "#4D4D4D", "River": "#1F6FB4",
            "Unannotated": "#FFFFFF"}
CLR = DISPLAY if USE_DISPLAY_PALETTE else RELEASED

ARCHS = ["U-Net (R18)", "UNetFormer (R18)", "SegFormer-B0", "SegNeXt-T", "TC-SegFormer"]
ACLR = {"U-Net (R18)": "#8C8C8C", "UNetFormer (R18)": "#6BAED6", "SegFormer-B0": "#2C7FB8",
        "SegNeXt-T": "#E8A33D", "TC-SegFormer": "#B2182B"}

C_BLOCK, C_RANDOM = "#B2182B", "#2166AC"
print("palette:", "display" if USE_DISPLAY_PALETTE else "released")

## 3 · Data

Every number for the plotted figures, with its source table in the comment. Unchanged.

In [ ]:
# ---------------- dataset (paper Table 1)
SHARE = {"Residential": 56.02, "Agricultural": 18.10, "UnusedLand": 12.97,
         "Forest": 10.39, "Road": 1.57, "River": 0.60}
TILES = {"Residential": 360, "Agricultural": 191, "UnusedLand": 225,
         "Forest": 132, "Road": 214, "River": 83}
N_TILES = 408

# Jeffries-Matusita, the pairs that are actually reported anywhere (0-2 scale)
JM_KNOWN = {
    frozenset(["UnusedLand", "Agricultural"]):  0.051,
    frozenset(["Residential", "UnusedLand"]):   0.204,
    frozenset(["Residential", "Agricultural"]): 0.254,
    frozenset(["Road", "River"]):               0.290,
    frozenset(["Road", "Forest"]):              1.322,
}
# NEEDS DATA: the other 10 pairs, from your dataset-characterisation notebook.

VAL_RESID_SHARE = [30.66, 64.49, 73.65]
VAL_RIVER_SHARE = [0.041, 0.865, 0.888]
VAL_RIVER_TILES = [3, 38, 42]

# ---------------- part 1 mIoU (paper Table 2)
MIOU_BLOCK = {"U-Net (R18)": [0.3213, 0.4516, 0.4228],
              "UNetFormer (R18)": [0.2795, 0.5069, 0.4736],
              "SegFormer-B0": [0.2877, 0.4657, 0.4351],
              "SegNeXt-T": [0.3345, 0.5701, 0.5122],
              "TC-SegFormer": [0.3403, 0.5582, 0.5061]}
MIOU_RANDOM = {"U-Net (R18)": [0.4533, 0.4465, 0.4484],
               "UNetFormer (R18)": [0.5263, 0.5007, 0.4905],
               "SegFormer-B0": [0.4948, 0.4736, 0.4611],
               "SegNeXt-T": [0.5646, 0.5498, 0.5450],
               "TC-SegFormer": [0.5525, 0.5650, 0.5717]}
SD_RATIO = {"U-Net (R18)": 19.5, "UNetFormer (R18)": 6.7, "SegFormer-B0": 5.6,
            "SegNeXt-T": 12.0, "TC-SegFormer": 11.7}

# ---------------- part 1 per class (results master Tables 4.3a / 4.3b)
PC_BLOCK = {"U-Net (R18)":      [0.7553, 0.4757, 0.1899, 0.5694, 0.3142, 0.0868],
            "UNetFormer (R18)": [0.7342, 0.4782, 0.2154, 0.5699, 0.3112, 0.2108],
            "SegFormer-B0":     [0.7093, 0.4935, 0.2411, 0.5899, 0.2461, 0.0971],
            "SegNeXt-T":        [0.7872, 0.5311, 0.2118, 0.5935, 0.3918, 0.3182],
            "TC-SegFormer":     [0.7698, 0.5171, 0.2428, 0.5886, 0.3603, 0.3305]}
PC_RANDOM = {"U-Net (R18)":      [0.8098, 0.4614, 0.2893, 0.6027, 0.3788, 0.1546],
             "UNetFormer (R18)": [0.8150, 0.5163, 0.3296, 0.6175, 0.3616, 0.3949],
             "SegFormer-B0":     [0.7969, 0.5287, 0.3814, 0.6672, 0.2886, 0.1960],
             "SegNeXt-T":        [0.8333, 0.5289, 0.3532, 0.6517, 0.4526, 0.4990],
             "TC-SegFormer":     [0.8354, 0.5324, 0.3978, 0.6690, 0.4285, 0.5155]}

# ---------------- ablation (results master Tables 6.1, 6.3)
ABL_ORDER = ["full", "- detail path", "- sampler", "- soft-clDice", "- CSA", "backbone"]
ABL_MIOU  = {"full": 0.5582, "- detail path": 0.5449, "- sampler": 0.5509,
             "- soft-clDice": 0.5522, "- CSA": 0.5564, "backbone": 0.4657}
ABL_PERCLASS = {
    "- detail path":  {"Residential": +0.0003, "Agricultural": +0.0014, "UnusedLand": -0.0180,
                       "Forest": +0.0054, "Road": -0.0295, "River": -0.0392},
    "- sampler":      {"Residential": +0.0021, "Agricultural": -0.0123, "UnusedLand": -0.0262,
                       "Forest": +0.0035, "Road": -0.0075, "River": -0.0030},
    "- soft-clDice":  {"Residential": -0.0015, "Agricultural": +0.0020, "UnusedLand": -0.0004,
                       "Forest": -0.0038, "Road": -0.0159, "River": -0.0162},
    "- CSA":          {"Residential": -0.0020, "Agricultural": -0.0049, "UnusedLand": +0.0024,
                       "Forest": +0.0000, "Road": +0.0013, "River": -0.0077}}
ABL_COL = {"full": "#4D4D4D", "- detail path": "#B2182B", "- sampler": "#E8A33D",
           "- soft-clDice": "#2C7FB8", "- CSA": "#BDBDBD", "backbone": "#9ECAE1"}
DEFENSIBLE = 0.005

# ---------------- LoveDA (results master Table 7.2)
LOVEDA_CLASSES = ["Background", "Building", "Road", "Water", "Barren", "Forest", "Agriculture"]
LOVEDA_B0 = [0.4436, 0.5051, 0.5310, 0.6038, 0.3149, 0.3870, 0.5255]
LOVEDA_TC = [0.4219, 0.5348, 0.5468, 0.6480, 0.2304, 0.3652, 0.5552]

# ---------------- Stage 1 (results master Tables 8.1, 8.3)
S1_KL   = {"binary label code": 0.79364, "one-hot": 0.00001}
S1_FRAG = {"binary label code": {"Road": 6.62, "River": 6.07},
           "one-hot":           {"Road": 1.54, "River": 1.52}}
S1_COL  = {"binary label code": "#B2182B", "one-hot": "#1A7A3C"}

# ---------------- Stage 2 selection trajectory (results master Table 9.2)
S2_STEP   = [500, 1000, 1500, 2000, 3000, 3500, 4000, 4500, 5000, 5500, 6000, 6500, 7000]
S2_LAYOUT = [0.2415, 0.3131, 0.2726, 0.2800, 0.3058, 0.3907, 0.3741, 0.3799,
             0.3664, 0.3837, 0.3905, 0.4111, 0.4171]
S2_EPS_STEP = [1700, 7200]        # NEEDS DATA: full W&B trace; only endpoints are recorded
S2_EPS_MSE  = [0.1158, 0.1186]

# ---------------- Stage 3 (results master Tables 10.1, 10.2)
S3_SETS   = ["R1 (804)", "R2 (1608)", "R3 (2412)"]
S3_LAYOUT = [0.4395, 0.4375, 0.4337]
S3_CEIL   = 0.4646
S3_MEM    = {"Inception feature": [0.99, 1.00, 1.01],
             "Raw pixel":         [0.89, 0.90, 0.90],
             "Mask":              [1.39, 1.34, 1.42]}
S3_COL    = ["#9ECAE1", "#4292C6", "#08519C"]

# ---------------- Stage 4 (results master Tables 11.4, 11.5c, 11.5e-i, 11.5e-ii)
S4_MIOU  = {"U-Net (R18)": (-0.0057, -0.0130, +0.0021, False),
            "UNetFormer (R18)": (-0.0159, -0.0292, -0.0022, True),
            "SegFormer-B0": (-0.0071, -0.0180, +0.0037, False),
            "SegNeXt-T": (-0.0364, -0.0496, -0.0214, True),
            "TC-SegFormer": (-0.0047, -0.0170, +0.0078, False)}
S4_RIVER = {"U-Net (R18)": (-0.0406, -0.0520, -0.0291, True),
            "UNetFormer (R18)": (-0.1008, -0.1225, -0.0778, True),
            "SegFormer-B0": (-0.0324, -0.0457, -0.0209, True),
            "SegNeXt-T": (-0.1072, -0.1310, -0.0839, True),
            "TC-SegFormer": (-0.0039, -0.0416, +0.0351, False)}
S4_ROAD  = {"U-Net (R18)": (-0.0475, -0.0579, -0.0370, True),
            "UNetFormer (R18)": (-0.0305, -0.0408, -0.0193, True),
            "SegFormer-B0": (-0.0076, -0.0165, +0.0013, False),
            "SegNeXt-T": (-0.0671, -0.0852, -0.0493, True),
            "TC-SegFormer": (-0.0021, -0.0153, +0.0120, False)}
S4_MEAN = {"mIoU": -0.0140, "River": -0.0570, "Road": -0.0310}

S4_CLASS_ORDER = ["River", "Road", "UnusedLand", "Residential", "Forest", "Agricultural"]
S4_DELTA = {"U-Net (R18)":      [-0.0406, -0.0475, +0.0434, -0.0168, +0.0111, +0.0162],
            "UNetFormer (R18)": [-0.1008, -0.0305, +0.0394, -0.0137, -0.0013, +0.0118],
            "SegFormer-B0":     [-0.0324, -0.0076, +0.0130, -0.0081, -0.0169, +0.0095],
            "SegNeXt-T":        [-0.1072, -0.0671, -0.0196, -0.0145, -0.0089, -0.0013],
            "TC-SegFormer":     [-0.0039, -0.0021, -0.0057, +0.0028, -0.0130, -0.0067]}
S4_SIG = {"U-Net (R18)":      [1, 1, 1, 1, 0, 1],
          "UNetFormer (R18)": [1, 1, 1, 1, 0, 0],
          "SegFormer-B0":     [1, 0, 0, 0, 0, 0],
          "SegNeXt-T":        [1, 1, 0, 1, 0, 0],
          "TC-SegFormer":     [0, 0, 0, 0, 0, 0]}
S4_DELTA_MEAN = [-0.0570, -0.0310, +0.0141, -0.0101, -0.0058, +0.0059]

# mechanism (results master Table 11.5d)
FIDELITY = {"River":        (-0.0570, 0.0567, 0.424),
            "Road":         (-0.0310, 0.2187, 0.731),
            "Residential":  (-0.0101, 0.8143, 1.017),
            "Forest":       (-0.0058, 0.6757, 0.904),
            "Agricultural": (+0.0059, 0.5436, 0.966),
            "UnusedLand":   (+0.0141, 0.2950, 1.229)}

FWIOU = {"U-Net (R18)": (-0.0044, -0.0057), "UNetFormer (R18)": (-0.0045, -0.0159),
         "SegFormer-B0": (-0.0039, -0.0071), "SegNeXt-T": (-0.0143, -0.0364),
         "TC-SegFormer": (-0.0008, -0.0047)}
CLDICE = {"U-Net (R18)": (-0.0118, -0.0282, 408, 408),
          "UNetFormer (R18)": (-0.0106, -0.0381, 399, 408),
          "SegFormer-B0": (-0.0079, -0.0222, 399, 408),
          "SegNeXt-T": (-0.0135, -0.0329, 363, 370),
          "TC-SegFormer": (-0.0046, -0.0890, 234, 334)}

EFF = {"U-Net (R18)": 12.458, "UNetFormer (R18)": 11.725, "SegFormer-B0": 3.716,
       "SegNeXt-T": 4.226, "TC-SegFormer": 3.794}

GSD_CONTROLS = [("Penalty area width", 40.32, 0.2998), ("Penalty area depth", 16.50, 0.2989),
                ("Goal area width", 18.32, 0.3061), ("Goal area depth", 5.50, 0.3087)]
GSD_COMBINED, GSD_HALFRANGE = 0.3016, 0.0049

print("data loaded")

## 4 · Block-diagram toolkit

Unchanged.

In [ ]:
# ---- the only two colour families in any diagram
STYLE = {
    "base":  dict(fc="#F2F2F2", ec="#8C8C8C", lw=0.9),   # unchanged / pretrained / frozen
    "added": dict(fc="#DCE9F7", ec="#2B6CB0", lw=1.1),   # your contribution
    "emph":  dict(fc="#F7DDD9", ec="#B2182B", lw=1.2),   # output / headline node
    "group": dict(fc="none",    ec="#BFBFBF", lw=0.8),   # container
}
TXT = "#1A1A1A"

def diagram_axes(ax, W, H):
    ax.set_xlim(0, W); ax.set_ylim(H, 0)
    ax.set_aspect("equal"); ax.axis("off")
    return ax

def panel_height(W_in, W_units, H_units):
    return W_in * H_units / W_units

def box(ax, x, y, w, h, title=None, lines=None, kind="base", ts=7.0, ls=6.2,
        round_size=3, title_weight="normal", zorder=2):
    st = STYLE[kind]
    ax.add_patch(FancyBboxPatch((x, y), w, h,
                                boxstyle="round,pad=0,rounding_size=" + str(round_size),
                                fc=st["fc"], ec=st["ec"], lw=st["lw"], zorder=zorder))
    body = list(lines) if lines else []
    n = (1 if title else 0) + len(body)
    if n == 0:
        return
    gap = h / (n + 0.6)
    y0 = y + h / 2 - (n - 1) * gap / 2
    k = 0
    if title:
        ax.text(x + w / 2, y0, title, ha="center", va="center", fontsize=ts,
                color=TXT, weight=title_weight, zorder=zorder + 1)
        k = 1
    for i, t in enumerate(body):
        ax.text(x + w / 2, y0 + (i + k) * gap, t, ha="center", va="center",
                fontsize=ls, color="#333333", zorder=zorder + 1)

def group(ax, x, y, w, h, label, ls=7.0):
    st = STYLE["group"]
    ax.add_patch(FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0,rounding_size=4",
                                fc="none", ec=st["ec"], lw=st["lw"], ls=(0, (4, 3)), zorder=1))
    ax.text(x + 6, y + 9, label, ha="left", va="center", fontsize=ls, color="#666666",
            style="italic", zorder=2)

def _route(p0, p1, route):
    # explicit polyline: predictable, unlike connectionstyle="angle" which fails on
    # parallel angles. "hv" turns horizontally first, "vh" vertically first.
    if route == "hv":
        return [p0, (p1[0], p0[1]), p1]
    if route == "vh":
        return [p0, (p0[0], p1[1]), p1]
    return [p0, p1]

def arrow(ax, p0, p1, style="solid", route="straight", label=None,
          loff=(0, -6), fs=6.0, color="#4D4D4D", zorder=4):
    pts = _route(p0, p1, route)
    ls = "-" if style == "solid" else (0, (3.5, 2.5))
    lw = 1.0 if style == "solid" else 0.85
    for a, b in zip(pts[:-2], pts[1:-1]):
        ax.plot([a[0], b[0]], [a[1], b[1]], color=color, lw=lw, ls=ls,
                solid_capstyle="round", zorder=zorder)
    ax.add_patch(FancyArrowPatch(pts[-2], pts[-1], arrowstyle="-|>", mutation_scale=6,
                                 lw=lw, ls=ls, color=color,
                                 connectionstyle="arc3,rad=0",
                                 shrinkA=0, shrinkB=1.5, zorder=zorder))
    if label:
        cx = sum(q[0] for q in pts) / len(pts) + loff[0]
        cy = sum(q[1] for q in pts) / len(pts) + loff[1]
        ax.text(cx, cy, label, ha="center", va="center", fontsize=fs, color="#666666",
                zorder=zorder + 1, bbox=dict(fc="white", ec="none", pad=0.8))

def circ(ax, cx, cy, r, symbol="+", kind="emph", fs=9):
    st = STYLE[kind]
    ax.add_patch(Circle((cx, cy), r, fc=st["fc"], ec=st["ec"], lw=st["lw"], zorder=3))
    ax.text(cx, cy, symbol, ha="center", va="center", fontsize=fs, color=TXT, zorder=4)

def note(ax, x, y, text, fs=5.8, ha="left"):
    ax.text(x, y, text, ha=ha, va="center", fontsize=fs, color="#8C8C8C", style="italic")

print("toolkit loaded")

## 5 · Find the dataset, decode the masks, count every pixel

This is the cell that fixes the DEMO tiles. It scans `/kaggle/input` for folders of
images and masks whatever they are named, measures which colour encoding your masks
actually use instead of assuming one, and then counts the class pixels in every tile.
The measured class shares are printed next to Table 1's — if those two columns agree,
everything downstream is reading your data correctly.

In [ ]:
import random, glob, json, csv
from PIL import Image

rng = random.Random(SEED)
RASTER = (".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp")
CACHE = Path("figs/_cache"); CACHE.mkdir(parents=True, exist_ok=True)


def listdir(d):
    if not d or d == "auto" or not Path(d).exists():
        return []
    out = []
    for p in sorted(Path(d).iterdir()):
        if p.suffix.lower() in RASTER:
            out.append(str(p))
    return out


def stem(p):
    return Path(p).stem


def _digits(s):
    return "".join(ch for ch in str(s) if ch.isdigit())


def thumb(path, n=192, index_safe=True):
    """Nearest-neighbour subsample. Never interpolates, so class colours survive."""
    try:
        im = Image.open(path)
        if im.mode not in ("RGB", "L", "P", "I;16", "I"):
            im = im.convert("RGB")
        if im.mode == "P":
            im = im.convert("RGB")
        if n and max(im.size) > n:
            im = im.resize((n, n), Image.NEAREST)
        a = np.array(im)
        if a.ndim == 3:
            a = a[..., :3]
        return a
    except Exception:
        return None


# ------------------------------------------------------------------ 1. find the folders
def _candidate_dirs(roots, max_depth=6, min_files=4):
    seen = {}
    skip = (str(OUT.resolve()), "_cache", "_preds", ".git", "__pycache__")
    for root in roots:
        root = Path(root)
        if not root.exists():
            continue
        for p in root.rglob("*"):
            try:
                if not p.is_dir():
                    continue
                rp = str(p.resolve())
                if any(t in rp for t in skip):      # never treat our own output as data
                    continue
                if len(p.relative_to(root).parts) > max_depth:
                    continue
                n = sum(1 for q in p.iterdir() if q.suffix.lower() in RASTER)
                if n >= min_files:
                    seen[str(p)] = n
            except Exception:
                continue
    return seen


def _looks_like_mask(paths, k=3):
    """A mask has a handful of colours. A photograph has thousands."""
    uniq = set()
    for p in paths[:k]:
        a = thumb(p, 96)
        if a is None:
            continue
        if a.ndim == 2:
            uniq |= set(np.unique(a).tolist())
        else:
            uniq |= set(map(tuple, np.unique(a.reshape(-1, a.shape[-1]), axis=0).tolist()))
    return (len(uniq) <= 24), len(uniq)


def discover():
    """Find image / mask / generated folders wherever Kaggle mounted them."""
    roots = [Path("/kaggle/input"), Path("input"), Path("data"), Path(".")]
    cands = _candidate_dirs(roots)
    if not cands:
        return {}, []
    info, log = {}, []
    for d, n in sorted(cands.items()):
        ps = listdir(d)
        ismask, nu = _looks_like_mask(ps)
        low = d.lower()
        fake = any(t in low for t in ("gen", "fake", "synth", "stage3", "stage-3"))
        named_mask = any(t in low for t in ("mask", "label", "annot", "gt", "seg"))
        info[d] = dict(n=n, mask=(ismask or named_mask), fake=fake, colours=nu, paths=ps)
        log.append((d, n, nu, "mask" if (ismask or named_mask) else "image",
                    "generated" if fake else "real"))

    def best(is_mask, is_fake):
        pool = [(v["n"], d) for d, v in info.items()
                if v["mask"] == is_mask and v["fake"] == is_fake]
        return max(pool)[1] if pool else ""

    def pair_with(img_dir, is_fake):
        """Among mask folders, take the one whose file stems overlap the images most."""
        if not img_dir:
            return ""
        want = {_digits(stem(p)) for p in info[img_dir]["paths"][:60]}
        scored = []
        for d, v in info.items():
            if not v["mask"] or v["fake"] != is_fake:
                continue
            have = {_digits(stem(p)) for p in v["paths"][:60]}
            scored.append((len(want & have), v["n"], d))
        return max(scored)[2] if scored else ""

    found = {}
    found["img"] = best(False, False)
    found["mask"] = pair_with(found["img"], False)
    found["fake_img"] = best(False, True)
    found["fake_mask"] = pair_with(found["fake_img"], True)
    return found, log


_auto, _log = ({}, []) if all(v != "auto" for v in (IMG_DIR, MASK_DIR, FAKE_IMG, FAKE_MASK)) \
    else discover()

IMG_DIR = _auto.get("img", "") if IMG_DIR == "auto" else IMG_DIR
MASK_DIR = _auto.get("mask", "") if MASK_DIR == "auto" else MASK_DIR
FAKE_IMG = _auto.get("fake_img", "") if FAKE_IMG == "auto" else FAKE_IMG
FAKE_MASK = _auto.get("fake_mask", "") if FAKE_MASK == "auto" else FAKE_MASK

REAL_IMGS, REAL_MASKS = listdir(IMG_DIR), listdir(MASK_DIR)
FAKE_IMGS, FAKE_MASKS = listdir(FAKE_IMG), listdir(FAKE_MASK)
PREDS = listdir(PRED_DIR)
DEMO = len(REAL_IMGS) == 0

if _log:
    print("folders scanned under /kaggle/input")
    print("   %-58s %6s %8s %-6s %s" % ("path", "files", "colours", "kind", "source"))
    for d, n, nu, kind, srcs in _log[:24]:
        print("   %-58s %6d %8d %-6s %s" % (d[-58:], n, nu, kind, srcs))
    if len(_log) > 24:
        print("   ... and", len(_log) - 24, "more")
print()
print("images     :", len(REAL_IMGS), "  <-", IMG_DIR or "not found")
print("masks      :", len(REAL_MASKS), "  <-", MASK_DIR or "not found")
print("fake images:", len(FAKE_IMGS), "  <-", FAKE_IMG or "not found")
print("fake masks :", len(FAKE_MASKS), "  <-", FAKE_MASK or "not found")
print("predictions:", len(PREDS), "  <-", PRED_DIR or "none supplied")


# ------------------------------------------------------- 2. how are the masks encoded?
def hex2rgb(h):
    return np.array([int(h[i:i + 2], 16) for i in (1, 3, 5)], np.uint8)


def _key(rgb):
    return int(rgb[0]) << 16 | int(rgb[1]) << 8 | int(rgb[2])


def detect_encoding(paths, k=12):
    """Measure the colours actually present and match them to the class palette.

    Tries the released palette as RGB, the same palette byte-reversed (which is what a
    BGR-written file looks like when read back as RGB) and the display palette, and
    keeps whichever explains the most pixels. Index-coded masks are detected too.
    """
    if not paths:
        return dict(mode="none", cmap={}, cover=0.0, note="no masks")
    counts = {}
    gray_levels = set()
    total = 0
    for p in rng.sample(paths, min(k, len(paths))):
        a = thumb(p, 160)
        if a is None:
            continue
        if a.ndim == 2 or (a.ndim == 3 and np.all(a[..., 0] == a[..., 1])
                           and np.all(a[..., 1] == a[..., 2])):
            g = a if a.ndim == 2 else a[..., 0]
            gray_levels |= set(np.unique(g).tolist())
            total += g.size
            for v in np.unique(g):
                counts[("g", int(v))] = counts.get(("g", int(v)), 0) + int((g == v).sum())
            continue
        flat = a.reshape(-1, 3)
        total += len(flat)
        cols, cnt = np.unique(flat, axis=0, return_counts=True)
        for c, n in zip(cols, cnt):
            counts[("c", _key(c))] = counts.get(("c", _key(c)), 0) + int(n)

    grays = {v: n for (t, v), n in counts.items() if t == "g"}
    cols = {v: n for (t, v), n in counts.items() if t == "c"}

    # index-coded: a handful of grey levels, all small or evenly spaced
    if grays and len(grays) <= 8 and not cols:
        levels = sorted(grays)
        cmap = {}
        if max(levels) < len(CLASS_ORDER_IN_MASKS):        # already class codes 0..5
            for lv in levels:
                cmap[lv] = CLASS_ORDER_IN_MASKS[lv]
        else:                                              # spaced grey levels, by rank
            for i, lv in enumerate(levels[:len(CLASS_ORDER_IN_MASKS)]):
                cmap[lv] = CLASS_ORDER_IN_MASKS[i]
        cover = sum(grays[l] for l in cmap) / max(total, 1)
        return dict(mode="index", cmap=cmap, cover=cover,
                    note="levels " + str(levels) + " -> " + ", ".join(cmap.values()))

    trials = {}
    for name, pal in (("released", RELEASED), ("released-byte-reversed", RELEASED),
                      ("display", DISPLAY)):
        cmap, cov = {}, 0
        for cls, hx in pal.items():
            rgb = hex2rgb(hx)
            if name.endswith("reversed"):
                rgb = rgb[::-1]
            kk = _key(rgb)
            if kk in cols:
                cmap[kk] = cls
                cov += cols[kk]
        trials[name] = (cov / max(total, 1), cmap)
    best = max(trials.items(), key=lambda kv: kv[1][0])
    return dict(mode="colour:" + best[0], cmap=best[1][1], cover=best[1][0],
                note=str(len(best[1][1])) + " of 7 palette colours found in the files")


ENC = detect_encoding(REAL_MASKS) if REAL_MASKS else dict(mode="none", cmap={}, cover=0.0,
                                                          note="no masks found")
ENC_FAKE = detect_encoding(FAKE_MASKS) if FAKE_MASKS else dict(mode="none", cmap={},
                                                               cover=0.0, note="")
print("\nmask encoding (real) :", ENC["mode"], "| pixels explained %.1f%%" % (100 * ENC["cover"]),
      "|", ENC["note"])
if FAKE_MASKS:
    print("mask encoding (fake) :", ENC_FAKE["mode"],
          "| pixels explained %.1f%%" % (100 * ENC_FAKE["cover"]), "|", ENC_FAKE["note"])
if REAL_MASKS and ENC["cover"] < 0.5:
    print("  WARNING: less than half the mask pixels matched a known class colour.")
    print("  Check MASK_DIR, or set CLASS_ORDER_IN_MASKS if your masks are index-coded.")


def _lut(enc):
    """colour key or grey level -> class index in CLASSES order (255 = unannotated)."""
    out = {}
    for k, cls in enc["cmap"].items():
        out[k] = CLASSES.index(cls) if cls in CLASSES else 255
    return out


LUT, LUT_FAKE = _lut(ENC), _lut(ENC_FAKE)


def to_index(arr, enc=None, lut=None):
    """HxWx3 or HxW -> HxW int16 of class indices, 255 where unannotated/unknown."""
    if arr is None:
        return None
    enc = enc or ENC
    lut = LUT if lut is None else lut
    if enc["mode"] == "index":
        g = arr if arr.ndim == 2 else arr[..., 0]
        out = np.full(g.shape, 255, np.int16)
        for lv, ci in lut.items():
            out[g == lv] = ci
        return out
    a = arr if arr.ndim == 3 else np.dstack([arr] * 3)
    keys = (a[..., 0].astype(np.int32) << 16) | (a[..., 1].astype(np.int32) << 8) | a[..., 2]
    out = np.full(keys.shape, 255, np.int16)
    for k, ci in lut.items():
        out[keys == k] = ci
    return out


# --------------------------------------------------------- 3. the census: every pixel
CENSUS_PATH = CACHE / "tile_census.csv"


def build_census(force=False, n=192):
    """Class pixel fractions for every annotated tile. Cached, so it runs once."""
    if not REAL_MASKS:
        return {}
    sig = str(len(REAL_MASKS)) + "|" + ENC["mode"] + "|" + str(n)
    if CENSUS_PATH.exists() and not force:
        try:
            rows = list(csv.DictReader(open(CENSUS_PATH)))
            if rows and rows[0].get("_sig") == sig:
                return {r["stem"]: {c: float(r[c]) for c in CLASSES} for r in rows}
        except Exception:
            pass
    stats = {}
    for i, p in enumerate(REAL_MASKS):
        idx = to_index(thumb(p, n))
        if idx is None:
            continue
        tot = idx.size
        stats[stem(p)] = {c: float((idx == k).sum()) / tot for k, c in enumerate(CLASSES)}
        if (i + 1) % 100 == 0:
            print("   census %d / %d" % (i + 1, len(REAL_MASKS)))
    with open(CENSUS_PATH, "w", newline="") as fh:
        w = csv.DictWriter(fh, fieldnames=["stem", "_sig"] + CLASSES)
        w.writeheader()
        for s, d in stats.items():
            r = dict(d); r["stem"] = s; r["_sig"] = sig
            w.writerow(r)
    return stats


TILE_STATS = build_census()

if TILE_STATS:
    corpus = {c: float(np.mean([v[c] for v in TILE_STATS.values()])) for c in CLASSES}
    tot = sum(corpus.values()) or 1.0
    print("\nclass share measured from your masks, against the value in the paper (Table 1):")
    print("   %-14s %10s %10s" % ("class", "measured", "paper"))
    for c in CLASSES:
        print("   %-14s %9.2f%% %9.2f%%" % (c, 100 * corpus[c] / tot, SHARE[c]))
    print("   If these two columns disagree badly the mask encoding above is wrong;")
    print("   nothing downstream will be right until they agree.")
elif not DEMO:
    print("\nno masks read - the montages will use images only")
if DEMO:
    print("\nNO DATASET FOUND. The montages will draw synthetic DEMO tiles, which is what")
    print("you saw last time. Attach the dataset, or set IMG_DIR / MASK_DIR by hand.")

## 6 · Montage toolkit

Same as before, except that masks are decoded from what is measured in the file
rather than from an assumed palette, so index-coded masks (the generated ones) and
byte-reversed masks both render correctly.

In [ ]:
import matplotlib.image as mpimg


def _u8(a):
    if a.dtype != np.uint8:
        a = (np.clip(a, 0, 1) * 255).astype(np.uint8) if a.max() <= 1.0 else a.astype(np.uint8)
    return a[..., :3] if a.ndim == 3 else np.dstack([a] * 3)


def load(path):
    try:
        return _u8(mpimg.imread(path))
    except Exception:
        return None


def by_stem(paths, s):
    for p in paths:
        if stem(p) == s:
            return p
    for p in paths:                      # tolerate image123 / mask123 naming
        if _digits(stem(p)) == _digits(s) and s != "":
            return p
    return None


# ---- every candidate way a six-class mask can be stored, tried against the actual file
def _colour_lut(pal, reverse):
    out = {}
    for cls, hx in pal.items():
        rgb = hex2rgb(hx)
        if reverse:
            rgb = rgb[::-1]
        out[_key(rgb)] = CLASSES.index(cls) if cls in CLASSES else 255
    return out


_CANDIDATES = [("released", _colour_lut(RELEASED, False)),
               ("released-byte-reversed", _colour_lut(RELEASED, True)),
               ("display", _colour_lut(DISPLAY, False)),
               ("display-byte-reversed", _colour_lut(DISPLAY, True))]


def mask_to_index(mask):
    """Decode any mask array to class indices by measuring it, not by assuming."""
    if mask is None:
        return None, 0.0
    a = mask if mask.ndim == 3 else np.dstack([mask] * 3)
    grayish = np.all(a[..., 0] == a[..., 1]) and np.all(a[..., 1] == a[..., 2])
    if grayish and len(np.unique(a[..., 0])) <= 8:
        g = a[..., 0]
        levels = sorted(np.unique(g).tolist())
        out = np.full(g.shape, 255, np.int16)
        by_value = max(levels) < len(CLASS_ORDER_IN_MASKS)   # already class codes 0..5
        for i, lv in enumerate(levels[:len(CLASS_ORDER_IN_MASKS)]):
            cls = CLASS_ORDER_IN_MASKS[lv if by_value else i]
            out[g == lv] = CLASSES.index(cls) if cls in CLASSES else 255
        return out, float((out != 255).mean())
    keys = (a[..., 0].astype(np.int32) << 16) | (a[..., 1].astype(np.int32) << 8) | a[..., 2]
    best, best_cov = None, -1.0
    for _, lut in _CANDIDATES:
        out = np.full(keys.shape, 255, np.int16)
        for k, ci in lut.items():
            out[keys == k] = ci
        cov = float((out != 255).mean())
        if cov > best_cov:
            best, best_cov = out, cov
    return best, best_cov


def recolour(mask):
    """Repaint a mask in the figure palette. Pixels that match no class are left alone."""
    if mask is None:
        return None
    idx, cov = mask_to_index(mask)
    if idx is None or cov < 0.25:
        return mask
    out = mask.copy() if mask.ndim == 3 else np.dstack([mask] * 3)
    for k, c in enumerate(CLASSES):
        out[idx == k] = hex2rgb(CLR[c])
    return out


def overlay(img, mask, alpha=0.45):
    if img is None or mask is None:
        return None
    if img.shape[:2] != mask.shape[:2]:
        mask = np.array(Image.fromarray(mask).resize((img.shape[1], img.shape[0]),
                                                     Image.NEAREST))
    return (img.astype(np.float32) * (1 - alpha) + mask.astype(np.float32) * alpha).astype(np.uint8)


def crop(a, box_):
    if a is None or box_ is None:
        return a
    x0, y0, x1, y1 = box_
    return a[y0:y1, x0:x1]


def demo_tile(seed, kind="image", n=512):
    # obviously synthetic, so it can never be mistaken for data
    r = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:n, 0:n]
    lab = np.zeros((n, n), int)
    for c in range(1, 6):
        cy, cx = r.integers(0, n, 2)
        lab[(yy - cy) ** 2 + (xx - cx) ** 2 < r.integers(40, 130) ** 2] = c
    lab[np.abs(yy - xx + r.integers(-90, 90)) < 4] = 4
    if kind == "mask":
        out = np.zeros((n, n, 3), np.uint8)
        for c, k in enumerate(CLASSES):
            out[lab == c] = hex2rgb(CLR[k])
        return out
    base = np.zeros((n, n, 3), np.uint8)
    for c, k in enumerate(CLASSES):
        base[lab == c] = (hex2rgb(CLR[k]).astype(int) * 0.45 + 110).astype(np.uint8)
    return np.clip(base + r.integers(-16, 16, (n, n, 3)), 0, 255).astype(np.uint8)


def show(ax, arr, slot="", demo=False):
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_linewidth(0.5); sp.set_color("0.6")
    if arr is None:
        ax.add_patch(Rectangle((0, 0), 1, 1, transform=ax.transAxes, fc="#F4F4F4",
                               ec="#B2182B", lw=0.8))
        ax.text(0.5, 0.5, MISSING_TEXT.get(slot, "not available"), transform=ax.transAxes,
                ha="center", va="center", fontsize=6.2, color="#B2182B")
        return
    ax.imshow(arr, interpolation="nearest")     # never blend a mask
    if demo:
        ax.text(0.03, 0.97, "DEMO", transform=ax.transAxes, ha="left", va="top",
                fontsize=6.5, color="#B2182B",
                bbox=dict(fc="white", ec="#B2182B", lw=0.5, pad=1.0))


MISSING_TEXT = {}     # filled by the inference cell so a blank panel says why


def class_legend(fig, ncol=7):
    return top_legend(fig, [Patch(fc=CLR[c], ec="0.4", lw=0.3) for c in CLASSES]
                      + [Patch(fc=CLR["Unannotated"], ec="0.4", lw=0.3)],
                      CLASSES + ["unannotated"], ncol=ncol)


def pick(paths, n, avoid=()):
    pool = [p for p in paths if p not in avoid]
    return [pool[i] for i in sorted(rng.sample(range(len(pool)), min(n, len(pool))))] \
        if pool else []


print("montage toolkit loaded")

## 7 · The locator map for Fig. 1a

In [ ]:
# ---------------------------------------------------------------------------------
#  Fig. 1a locator map, drawn - not a screenshot.
#
#  The national outline is Natural Earth 1:50m admin-0 (public domain), simplified to
#  202 vertices and embedded below, so the cell needs no internet and no QGIS. If you
#  would rather use your own satellite or QGIS export, set LOCATOR_IMAGE in the control
#  panel and it is used instead.
#
#  A basemap screenshot is deliberately NOT used: Elsevier wants a cited, licensable
#  base for any map figure, and Section 3.1 says the per-tile footprints were lost, so
#  the study area is shown as an approximate extent and nothing more.
# ---------------------------------------------------------------------------------

NEPAL = [
    (88.110,27.871), (88.150,27.843), (88.154,27.799), (88.147,27.749), (88.106,27.642),
    (88.068,27.567), (88.024,27.409), (87.984,27.134), (87.993,27.086), (88.111,26.928),
    (88.157,26.807), (88.162,26.725), (88.112,26.586), (88.055,26.430), (88.027,26.395),
    (87.995,26.382), (87.849,26.437), (87.749,26.429), (87.633,26.399), (87.513,26.405),
    (87.414,26.423), (87.287,26.360), (87.167,26.394), (87.090,26.433), (87.038,26.542),
    (87.016,26.555), (86.763,26.442), (86.701,26.435), (86.544,26.496), (86.414,26.556),
    (86.366,26.574), (86.242,26.598), (86.129,26.612), (86.007,26.649), (85.856,26.600),
    (85.795,26.604), (85.737,26.640), (85.707,26.713), (85.700,26.782), (85.648,26.829),
    (85.568,26.840), (85.456,26.797), (85.293,26.741), (85.240,26.750), (85.192,26.767),
    (85.174,26.782), (85.152,26.847), (85.125,26.861), (85.087,26.863), (85.020,26.879),
    (84.937,26.927), (84.685,27.041), (84.654,27.092), (84.655,27.204), (84.641,27.250),
    (84.610,27.299), (84.481,27.348), (84.230,27.428), (84.091,27.491), (84.025,27.462),
    (83.897,27.435), (83.829,27.378), (83.747,27.396), (83.552,27.456), (83.447,27.465),
    (83.384,27.445), (83.369,27.410), (83.290,27.371), (83.214,27.402), (83.064,27.445),
    (82.933,27.468), (82.733,27.519), (82.711,27.597), (82.677,27.673), (82.630,27.687),
    (82.451,27.672), (82.288,27.757), (82.112,27.865), (82.037,27.901), (81.988,27.914),
    (81.945,27.899), (81.897,27.874), (81.853,27.867), (81.757,27.914), (81.636,27.980),
    (81.486,28.062), (81.311,28.176), (81.239,28.241), (81.206,28.289), (81.169,28.335),
    (81.017,28.410), (80.896,28.469), (80.751,28.540), (80.726,28.554), (80.671,28.596),
    (80.587,28.650), (80.518,28.665), (80.496,28.636), (80.479,28.605), (80.419,28.612),
    (80.325,28.666), (80.227,28.723), (80.150,28.776), (80.071,28.830), (80.052,28.870),
    (80.085,28.994), (80.130,29.100), (80.170,29.124), (80.233,29.195), (80.256,29.318),
    (80.255,29.423), (80.317,29.572), (80.402,29.730), (80.549,29.900), (80.613,29.956),
    (80.684,29.994), (80.820,30.119), (80.848,30.140), (80.908,30.172), (80.966,30.180),
    (81.010,30.165), (81.056,30.099), (81.110,30.037), (81.177,30.040), (81.255,30.093),
    (81.417,30.338), (81.642,30.387), (81.855,30.362), (82.043,30.327), (82.099,30.245),
    (82.135,30.159), (82.159,30.115), (82.221,30.064), (82.487,29.942), (82.641,29.831),
    (82.854,29.683), (83.014,29.618), (83.155,29.613), (83.235,29.555), (83.355,29.439),
    (83.457,29.306), (83.583,29.184), (83.671,29.188), (83.790,29.227), (83.936,29.279),
    (84.022,29.254), (84.101,29.220), (84.128,29.156), (84.176,29.036), (84.229,28.912),
    (84.312,28.868), (84.411,28.804), (84.465,28.753), (84.651,28.660), (84.677,28.622),
    (84.714,28.596), (84.759,28.579), (84.797,28.560), (84.855,28.554), (85.069,28.610),
    (85.126,28.603), (85.159,28.592), (85.160,28.572), (85.121,28.484), (85.089,28.372),
    (85.122,28.316), (85.212,28.293), (85.411,28.276), (85.678,28.277), (85.759,28.221),
    (85.840,28.135), (85.922,27.990), (85.954,27.928), (85.995,27.910), (86.064,27.935),
    (86.075,27.995), (86.079,28.084), (86.137,28.114), (86.174,28.092), (86.218,28.022),
    (86.329,27.960), (86.409,27.929), (86.485,27.940), (86.517,27.964), (86.554,28.085),
    (86.614,28.103), (86.691,28.095), (86.720,28.071), (86.750,28.022), (86.842,27.999),
    (86.934,27.968), (87.020,27.929), (87.141,27.838), (87.291,27.822), (87.464,27.824),
    (87.555,27.822), (87.623,27.815), (87.683,27.821), (87.861,27.886), (87.933,27.891),
    (88.023,27.883), (88.110,27.871)
]

VALLEY_BBOX = (85.19, 27.58, 85.56, 27.85)      # approximate extent, lon0 lat0 lon1 lat1
VALLEY_CITIES = [("Kathmandu", 85.324, 27.717), ("Lalitpur", 85.325, 27.659),
                 ("Bhaktapur", 85.430, 27.671)]
MAP_SOURCE = "Base outline: Natural Earth 1:50m (public domain)"


def _aspect(lat):
    return 1.0 / max(np.cos(np.deg2rad(lat)), 1e-6)


def draw_locator(ax):
    """Country outline, the valley extent, and a scale bar - all vector, all offline."""
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_linewidth(0.5); sp.set_color("0.6")

    if LOCATOR_IMAGE and Path(LOCATOR_IMAGE).exists():
        im = load(LOCATOR_IMAGE)
        if im is not None:
            ax.imshow(im, interpolation="nearest")
            ax.set_anchor("N")
            return

    ax.set_xlim(0, 1); ax.set_ylim(0, 1)

    # ---- panel 1: Nepal, with the valley marked
    a1 = ax.inset_axes([0.04, 0.76, 0.92, 0.20])
    xs = [p[0] for p in NEPAL]; ys = [p[1] for p in NEPAL]
    a1.fill(xs, ys, fc="#EDEDED", ec="#6E6E6E", lw=0.7, zorder=1)
    cx = 0.5 * (VALLEY_BBOX[0] + VALLEY_BBOX[2]); cy = 0.5 * (VALLEY_BBOX[1] + VALLEY_BBOX[3])
    a1.plot([cx], [cy], marker="s", ms=3.2, mfc="#B2182B", mec="#B2182B", zorder=3)
    a1.annotate("Kathmandu Valley", (cx, cy), xytext=(cx + 0.9, cy + 1.9),
                fontsize=6.0, color="#B2182B", ha="left",
                arrowprops=dict(arrowstyle="-", lw=0.6, color="#B2182B",
                                shrinkA=0, shrinkB=1.5))
    a1.text(80.3, 26.6, "NEPAL", fontsize=6.4, color="#4D4D4D", style="italic")
    a1.set_xlim(79.8, 88.6); a1.set_ylim(25.8, 30.9)
    a1.set_aspect(_aspect(28.3)); a1.axis("off")

    # ---- panel 2: the valley itself
    a2 = ax.inset_axes([0.06, 0.34, 0.88, 0.38])
    x0, y0, x1, y1 = VALLEY_BBOX
    mx, my = 0.10 * (x1 - x0), 0.10 * (y1 - y0)
    a2.add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fc="#EAF1F8", ec="#2B6CB0",
                           lw=1.0, zorder=1))
    for nm, lo, la in VALLEY_CITIES:
        a2.plot([lo], [la], marker="o", ms=2.2, mfc="#333333", mec="#333333", zorder=3)
        a2.text(lo + 0.012, la, nm, fontsize=5.8, va="center", color="#333333", zorder=3)
    a2.set_xlim(x0 - mx, x1 + mx); a2.set_ylim(y0 - my, y1 + my)
    a2.set_aspect(_aspect(27.7))
    a2.set_xticks([x0, x1]); a2.set_yticks([y0, y1])
    a2.set_xticklabels(["%.2f E" % x0, "%.2f E" % x1], fontsize=5.6)
    a2.set_yticklabels(["%.2f N" % y0, "%.2f N" % y1], fontsize=5.6, rotation=90, va="center")
    a2.tick_params(length=1.5, width=0.5, pad=1.5)
    for sp in a2.spines.values():
        sp.set_linewidth(0.5); sp.set_color("0.6")

    # 5 km scale bar, in degrees of longitude at this latitude
    km = 5.0
    dx = km / (111.32 * np.cos(np.deg2rad(27.7)))
    bx, by = x0 + 0.06 * (x1 - x0), y0 + 0.07 * (y1 - y0)
    a2.plot([bx, bx + dx], [by, by], color="0.15", lw=1.6, solid_capstyle="butt", zorder=4)
    a2.text(bx + dx / 2, by + 0.018 * (y1 - y0), "5 km", ha="center", fontsize=5.6,
            color="0.15", zorder=4)
    a2.annotate("N", xy=(x1 - 0.05 * (x1 - x0), y1 - 0.20 * (y1 - y0)),
                xytext=(x1 - 0.05 * (x1 - x0), y1 - 0.42 * (y1 - y0)),
                ha="center", fontsize=6.2, color="0.15",
                arrowprops=dict(arrowstyle="-|>", lw=0.7, color="0.15", mutation_scale=5))

    ax.text(0.5, 0.27, "approximate extent of the\n202.4 km$^2$ annotated corpus",
            ha="center", va="top", fontsize=6.2, color="#333333")
    ax.text(0.5, 0.03, MAP_SOURCE, ha="center", va="bottom", fontsize=5.4, color="#8C8C8C",
            style="italic")


print("locator ready ·", len(NEPAL), "outline vertices ·",
      ("using " + LOCATOR_IMAGE) if LOCATOR_IMAGE else "drawn from the embedded outline")

## 8 · Which tile goes where

Chosen from the census. Manual picks in the control panel always win.

In [ ]:
# =====================================================================================
#  Which tile goes where, decided from the census rather than by eye.
#  Anything you set in the control panel wins; only empty slots are filled here, and
#  the score behind every automatic choice is printed so you can overrule it.
# =====================================================================================

def _f(s, c):
    return TILE_STATS.get(s, {}).get(c, 0.0)


def _rank(score, exclude=(), n=1):
    pool = [(score(s), s) for s in TILE_STATS if s not in exclude]
    pool = [p for p in pool if p[0] > 0]
    pool.sort(reverse=True)
    return pool[:n]


def _fold_blocks(stems, folds=3):
    """Sequence-block folds over the export order, as Section 3.4 defines them."""
    ordered = sorted(stems, key=lambda s: (int(_digits(s) or 0), s))
    n = len(ordered)
    size = n // folds
    return [ordered[k * size:(k + 1) * size if k < folds - 1 else n] for k in range(folds)]


FIG1_NOTES = ["dense historic core", "river corridor", "cultivated fringe"]
FIG9_ROWLAB = ["highest-scoring scene", "intermediate scene", "lowest-scoring scene"]
FIG11_ROWLAB = ["high fidelity", "mid fidelity", "low fidelity, river-bearing"]
FIG11_SCORES = [None, None, None]
SELECTION_LOG = []

if AUTO_PICK and TILE_STATS:
    # ---------------- Fig. 1: one dense core, one river corridor, one cultivated fringe
    if not FIG1_TILES:
        core = _rank(lambda s: _f(s, "Residential") - 0.5 * _f(s, "Forest"))
        river = _rank(lambda s: _f(s, "River") + 0.2 * _f(s, "Residential"),
                      exclude=[c[1] for c in core])
        fringe = _rank(lambda s: _f(s, "Agricultural") + 0.6 * _f(s, "UnusedLand")
                       - 0.8 * _f(s, "Residential"),
                       exclude=[c[1] for c in core] + [c[1] for c in river])
        chosen, notes = [], []
        for got, lab, cls in ((core, "dense historic core", "Residential"),
                              (river, "river corridor", "River"),
                              (fringe, "cultivated fringe", "Agricultural")):
            if got:
                s = got[0][1]
                chosen.append(s)
                notes.append(lab)
                SELECTION_LOG.append(("Fig. 1 " + lab, s,
                                      "%s %.1f%% of the tile" % (cls, 100 * _f(s, cls))))
        if len(chosen) == 3:
            FIG1_TILES, FIG1_NOTES = chosen, notes

    # ---------------- Fig. 6: the crop must hold a road AND a large area class
    if not FIG6_TILE:
        def road_and_block(s):
            road = min(_f(s, "Road") / 0.04, 1.0)
            area = min(max(_f(s, "Residential"), _f(s, "Agricultural")) / 0.30, 1.0)
            return road * area
        got = _rank(road_and_block)
        if got:
            FIG6_TILE = got[0][1]
            SELECTION_LOG.append(("Fig. 6 weight-field tile", FIG6_TILE,
                                  "road %.2f%%, residential %.1f%% - both present, which is "
                                  "the whole point of the panel"
                                  % (100 * _f(FIG6_TILE, "Road"),
                                     100 * _f(FIG6_TILE, "Residential"))))

    # a crop inside that tile, scored the same way on the full-resolution mask
    if FIG6_TILE and FIG6_CROP is None:
        mp = by_stem(REAL_MASKS, FIG6_TILE)
        if mp:
            full = load(mp)
            idx, _ = mask_to_index(full)
            if idx is not None:
                H, W = idx.shape
                side = max(256, min(H, W) // 3)
                best, bx = -1, None
                for yy in range(0, H - side + 1, max(side // 3, 1)):
                    for xx in range(0, W - side + 1, max(side // 3, 1)):
                        w = idx[yy:yy + side, xx:xx + side]
                        road = min(float((w == CLASSES.index("Road")).mean()) / 0.05, 1.0)
                        area = min(float((w == CLASSES.index("Residential")).mean()) / 0.35, 1.0)
                        sc = road * area
                        if sc > best:
                            best, bx = sc, (xx, yy, xx + side, yy + side)
                if bx and best > 0:
                    FIG6_CROP = bx
                    SELECTION_LOG.append(("Fig. 6 crop", str(bx),
                                          "best road-and-block window of %d px, score %.2f"
                                          % (side, best)))

    # ---------------- Fig. A.1(a): the tile holding the pitch
    if not FIGA1_TILE:
        if by_stem(REAL_IMGS, "image264"):
            FIGA1_TILE = "image264"
            SELECTION_LOG.append(("Fig. A.1(a) pitch tile", FIGA1_TILE,
                                  "the tile named in the scale-recovery appendix"))
        else:
            got = _rank(lambda s: _f(s, "UnusedLand"))
            if got:
                FIGA1_TILE = got[0][1]
                SELECTION_LOG.append(("Fig. A.1(a) pitch tile", FIGA1_TILE,
                                      "GUESS - largest open-ground tile. Check by eye and set "
                                      "FIGA1_TILE if the pitch is elsewhere"))

    # ---------------- Fig. 9: three validation scenes of the fold in question
    if not FIG9_SCENES:
        blocks = _fold_blocks(list(TILE_STATS))
        val = blocks[min(FOLD, len(blocks) - 1)] if SPLIT_MODE == "block" else list(TILE_STATS)
        val = [s for s in val if s in TILE_STATS]
        if val:
            # without model output the three rows cannot be "best / worst", so they are
            # chosen and labelled by what they contain instead. The inference cell
            # replaces this with a real ranking when the checkpoints load.
            rich = sorted(val, key=lambda s: -sum(1 for c in CLASSES if _f(s, c) > 0.02))
            riv = sorted(val, key=lambda s: -_f(s, "River"))
            den = sorted(val, key=lambda s: -_f(s, "Residential"))
            picks = []
            for cand in (den, riv, rich):
                for s in cand:
                    if s not in picks:
                        picks.append(s); break
            FIG9_SCENES = picks[:3]
            FIG9_ROWLAB = ["dense scene", "river-bearing scene", "mixed-class scene"]
            SELECTION_LOG.append(("Fig. 9 scenes", ", ".join(FIG9_SCENES),
                                  "fold-%d validation block; labelled by content until a "
                                  "checkpoint scores them" % FOLD))

# ---------------- Fig. 11: three generated pairs, spanning what can be measured
if AUTO_PICK and FAKE_MASKS and not FIG11_PAIRS:
    scored = []
    for p in FAKE_MASKS[:400]:
        idx, cov = mask_to_index(thumb(p, 192))
        if idx is None or cov < 0.2:
            continue
        riv = float((idx == CLASSES.index("River")).mean())
        scored.append((riv, stem(p)))
    if scored:
        scored.sort(reverse=True)
        with_river = [s for r, s in scored if r > 0.005]
        no_river = [s for r, s in scored if r <= 0.005]
        picks = []
        for pool in (no_river, no_river[len(no_river) // 2:] if no_river else [], with_river):
            for s in pool:
                if s not in picks:
                    picks.append(s); break
        while len(picks) < 3 and scored:
            s = scored[len(picks)][1]
            if s not in picks:
                picks.append(s)
        FIG11_PAIRS = picks[:3]
        FIG11_ROWLAB = ["no river in the layout", "no river in the layout",
                        "river-bearing layout"]
        SELECTION_LOG.append(("Fig. 11 pairs", ", ".join(FIG11_PAIRS),
                              "river share ranked over %d generated masks; rows are labelled "
                              "by content until the scorer runs" % len(scored)))

if SELECTION_LOG:
    print("automatic choices, and why:")
    for what, which, why in SELECTION_LOG:
        print("   %-26s %-22s %s" % (what, which, why))
else:
    print("nothing to choose automatically - either every slot is set by hand,"
          " or no dataset was found.")

## 9 · Model output for Figs. 9 and 11 (optional, needs internet)

In [ ]:
# =====================================================================================
#  Model output for Fig. 9 (two predictions per row) and Fig. 11 (the scorer read-back).
#
#  This is the only cell that needs the network. On Kaggle turn Internet ON in the
#  session settings, otherwise leave INFER = "off".
#
#  Nothing here is ever faked. If a checkpoint will not load, the status is printed,
#  the affected panel says what is missing, and the rest of the notebook is unaffected.
# =====================================================================================

PRED_DIR_OUT = Path("figs/_preds"); PRED_DIR_OUT.mkdir(parents=True, exist_ok=True)
INFER_STATUS = {"segformer-b0": "not attempted", "tc-segformer": "not attempted"}
_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
_STD = np.array([0.229, 0.224, 0.225], np.float32)


def _paint(idx):
    out = np.zeros(idx.shape + (3,), np.uint8)
    out[:] = hex2rgb(CLR["Unannotated"])
    for k, c in enumerate(CLASSES):
        out[idx == k] = hex2rgb(CLR[c])
    return out


def _save_pred(idx, name):
    p = PRED_DIR_OUT / (name + ".png")
    Image.fromarray(_paint(idx)).save(p)
    return str(p)


def _iou(pred, gt, k):
    p, g = (pred == k), (gt == k)
    u = np.logical_or(p, g).sum()
    return float(np.logical_and(p, g).sum()) / u if u else np.nan


def _miou(pred, gt):
    vals = [_iou(pred, gt, k) for k in range(len(CLASSES))]
    vals = [v for v in vals if not np.isnan(v)]
    return float(np.mean(vals)) if vals else np.nan


# ---------------------------------------------------------------- model construction
def _torch():
    import torch
    return torch


def build_b0(repo):
    """Your B0 checkpoint stores model_state plus the full HF model_config."""
    torch = _torch()
    from huggingface_hub import hf_hub_download
    from transformers import SegformerForSemanticSegmentation, SegformerConfig
    ck = torch.load(hf_hub_download(repo, "best.pt"), map_location="cpu", weights_only=False)
    cfg = ck.get("model_config")
    model = SegformerForSemanticSegmentation(SegformerConfig(**cfg)) if isinstance(cfg, dict) \
        else SegformerForSemanticSegmentation.from_pretrained(repo)
    sd = ck.get("model_state", ck.get("state_dict", ck))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    miss, unexp = model.load_state_dict(sd, strict=False)
    model.eval()
    return model, len(miss), len(unexp)


class TCSegFormerGuess(object):
    """The proposed model rebuilt from Section 4.1, used ONLY if its parameter names and
    shapes match your checkpoint exactly (the check below is strict). Your training
    notebook is the authority on those names; if they differ, assign BUILD_TC_MODEL in
    the control panel to your own class, or export predictions and set PRED_DIR."""

    @staticmethod
    def make(num_classes=6, backbone="nvidia/segformer-b0-finetuned-ade-512-512"):
        torch = _torch()
        import torch.nn as nn
        import torch.nn.functional as F
        from transformers import SegformerForSemanticSegmentation

        class TC(nn.Module):
            def __init__(self):
                super().__init__()
                self.seg = SegformerForSemanticSegmentation.from_pretrained(
                    backbone, num_labels=num_classes, ignore_mismatched_sizes=True)
                self.detail = nn.Sequential(
                    nn.Conv2d(3, 8, 3, 2, 1, bias=False), nn.GroupNorm(4, 8), nn.ReLU(True),
                    nn.Conv2d(8, 16, 3, 1, 1, bias=False), nn.GroupNorm(4, 16), nn.ReLU(True),
                    nn.Conv2d(16, 32, 3, 1, 1, bias=False), nn.GroupNorm(8, 32), nn.ReLU(True))
                self.gamma = nn.Parameter(torch.tensor(0.1))
                self.proj = nn.Conv2d(256, 64, 1)
                self.fuse = nn.Sequential(nn.Conv2d(96, 64, 3, 1, 1, bias=False),
                                          nn.GroupNorm(8, 64), nn.ReLU(True))
                self.refine = nn.Conv2d(64, num_classes, 1)

            def _decode(self, feats):
                # the all-MLP head, unrolled so the fused 256-channel feature at H/4 is
                # available to the residual branch as well as to the classifier
                head = self.seg.decode_head
                B = feats[0].shape[0]
                outs = []
                for f, mlp in zip(feats, head.linear_c):
                    h = mlp(f).permute(0, 2, 1).reshape(B, -1, f.shape[2], f.shape[3])
                    outs.append(F.interpolate(h, size=feats[0].shape[2:], mode="bilinear",
                                              align_corners=False))
                fused = head.linear_fuse(torch.cat(outs[::-1], dim=1))
                fused = head.batch_norm(fused)
                fused = head.activation(fused) if hasattr(head, "activation") else F.relu(fused)
                return fused, head.classifier(fused)

            def forward(self, x):
                feats = self.seg.segformer(pixel_values=x,
                                           output_hidden_states=True).hidden_states
                fused, base = self._decode(feats)                      # H/4
                p = F.interpolate(self.proj(fused), scale_factor=2, mode="bilinear",
                                  align_corners=False)                 # H/2
                d = self.detail(x)                                     # H/2
                res = self.refine(self.fuse(torch.cat([p, self.gamma * d], 1)))
                up = lambda t: F.interpolate(t, size=x.shape[-2:], mode="bilinear",
                                             align_corners=False)
                return up(base) + up(res)
        return TC()


def build_tc(repo):
    torch = _torch()
    from huggingface_hub import hf_hub_download
    ck = torch.load(hf_hub_download(repo, "best.pt"), map_location="cpu", weights_only=False)
    sd = ck.get("model_state", ck.get("state_dict", ck))
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    model = BUILD_TC_MODEL() if callable(BUILD_TC_MODEL) else TCSegFormerGuess.make()
    have = dict(model.state_dict())
    matched = sum(1 for k, v in sd.items() if k in have and tuple(have[k].shape) == tuple(v.shape))
    frac = matched / max(len(sd), 1)
    if frac < 0.98:
        prefixes = sorted({k.split(".")[0] for k in sd})
        raise RuntimeError(
            "checkpoint keys do not match the reconstructed model (%.0f%% matched).\n"
            "        top-level keys in your checkpoint: %s\n"
            "        Paste your own TC-SegFormer class into BUILD_TC_MODEL, or export the\n"
            "        predictions from your training notebook and point PRED_DIR at them."
            % (100 * frac, ", ".join(prefixes)))
    model.load_state_dict(sd, strict=False)
    model.eval()
    return model, 0, 0


def predict(model, img, size=512):
    torch = _torch()
    a = np.array(Image.fromarray(img).resize((size, size), Image.BILINEAR), np.float32) / 255.0
    x = torch.from_numpy(((a - _MEAN) / _STD).transpose(2, 0, 1))[None]
    with torch.no_grad():
        out = model(x)
        logits = out.logits if hasattr(out, "logits") else out
        if logits.shape[-1] != size:
            import torch.nn.functional as F
            logits = F.interpolate(logits, size=(size, size), mode="bilinear",
                                   align_corners=False)
    return logits.argmax(1)[0].cpu().numpy().astype(np.int16)


MODELS = {}
if INFER != "off" and (REAL_IMGS or FAKE_IMGS):
    for tag, repo, builder in (("segformer-b0", HF_B0, build_b0),
                               ("tc-segformer", HF_TC, build_tc)):
        try:
            m, miss, unexp = builder(repo)
            MODELS[tag] = m
            INFER_STATUS[tag] = "loaded from " + repo + \
                (" (%d missing, %d unexpected keys)" % (miss, unexp) if miss or unexp else "")
        except Exception as e:
            INFER_STATUS[tag] = "not loaded - " + str(e).split("\n")[0][:180]
else:
    for t in INFER_STATUS:
        INFER_STATUS[t] = "skipped (INFER = 'off')" if INFER == "off" else "skipped (no images)"

for t, s in INFER_STATUS.items():
    print("%-14s %s" % (t, s))
if INFER != "off" and not MODELS:
    print("\nNo checkpoint loaded. On Kaggle this is almost always the Internet switch:")
    print("Session options -> Internet -> On, then re-run. The rest of the notebook is fine")
    print("without it; only the prediction columns of Figs. 9 and 11 stay empty.")

# ------------------------------------------------------- rank and render what we can
if MODELS and FIG9_SCENES is not None and REAL_IMGS and REAL_MASKS:
    scorer = MODELS.get("tc-segformer") or MODELS.get("segformer-b0")
    blocks = _fold_blocks(list(TILE_STATS)) if TILE_STATS else []
    val = blocks[min(FOLD, len(blocks) - 1)] if blocks else [stem(p) for p in REAL_IMGS]
    val = [s for s in val if by_stem(REAL_IMGS, s) and by_stem(REAL_MASKS, s)]
    if len(val) > MAX_SCORE_SCENES:
        val = [val[i] for i in np.linspace(0, len(val) - 1, MAX_SCORE_SCENES).astype(int)]
    print("\nscoring", len(val), "validation scenes of fold", FOLD, "...")
    scores = []
    for s in val:
        img = load(by_stem(REAL_IMGS, s))
        gt, cov = mask_to_index(load(by_stem(REAL_MASKS, s)))
        if img is None or gt is None or cov < 0.2:
            continue
        gt = np.array(Image.fromarray(gt.astype(np.uint8)).resize((512, 512), Image.NEAREST))
        scores.append((_miou(predict(scorer, img), gt), s))
    scores = [(v, s) for v, s in scores if not np.isnan(v)]
    if len(scores) >= 3:
        scores.sort()
        chosen = [scores[-1][1], scores[len(scores) // 2][1], scores[0][1]]
        FIG9_SCENES = chosen
        FIG9_ROWLAB = ["highest-scoring scene", "intermediate scene", "lowest-scoring scene"]
        FIG9_PRED_B0, FIG9_PRED_TC = [], []
        for r, s in enumerate(chosen):
            img = load(by_stem(REAL_IMGS, s))
            for tag, lst in (("segformer-b0", FIG9_PRED_B0), ("tc-segformer", FIG9_PRED_TC)):
                if tag in MODELS:
                    nm = "pred_" + tag + "_" + s
                    _save_pred(predict(MODELS[tag], img), nm)
                    lst.append(nm)
        PRED_DIR = str(PRED_DIR_OUT)
        PREDS = listdir(PRED_DIR)
        print("Fig. 9 rows: best %.3f · median %.3f · worst %.3f mIoU"
              % (scores[-1][0], scores[len(scores) // 2][0], scores[0][0]))

if MODELS and FAKE_IMGS and FAKE_MASKS:
    scorer = MODELS.get("tc-segformer") or MODELS.get("segformer-b0")
    pool = [stem(p) for p in FAKE_MASKS][:120]
    vals = []
    for s in pool:
        gi, gm = by_stem(FAKE_IMGS, s), by_stem(FAKE_MASKS, s)
        if not gi or not gm:
            continue
        img = load(gi)
        m, cov = mask_to_index(load(gm))
        if img is None or m is None or cov < 0.2:
            continue
        m = np.array(Image.fromarray(m.astype(np.uint8)).resize((512, 512), Image.NEAREST))
        vals.append((_miou(predict(scorer, img), m), s, float((m == CLASSES.index("River")).mean())))
    vals = [v for v in vals if not np.isnan(v[0])]
    if len(vals) >= 3:
        vals.sort(reverse=True)
        river = [v for v in vals if v[2] > 0.005]
        low = river[-1] if river else vals[-1]
        rows = [vals[0], vals[len(vals) // 2], low]
        FIG11_PAIRS = [r[1] for r in rows]
        FIG11_SCORES = [r[0] for r in rows]
        FIG11_ROWLAB = ["high fidelity", "mid fidelity", "low fidelity, river-bearing"]
        FIG11_READBACK = []
        for r in rows:
            nm = "readback_" + r[1]
            _save_pred(predict(scorer, load(by_stem(FAKE_IMGS, r[1]))), nm)
            FIG11_READBACK.append(nm)
        PRED_DIR = str(PRED_DIR_OUT)
        PREDS = listdir(PRED_DIR)
        print("Fig. 11 rows: layout mIoU %.3f / %.3f / %.3f" % tuple(FIG11_SCORES))

MISSING_TEXT.update({
    "FIG9_PRED_B0[0]": "SegFormer-B0 output\nnot available",
    "FIG9_PRED_B0[1]": "SegFormer-B0 output\nnot available",
    "FIG9_PRED_B0[2]": "SegFormer-B0 output\nnot available",
    "FIG9_PRED_TC[0]": "proposed-model output\nnot available",
    "FIG9_PRED_TC[1]": "proposed-model output\nnot available",
    "FIG9_PRED_TC[2]": "proposed-model output\nnot available",
    "FIG11_READBACK[0]": "scorer read-back\nnot available",
    "FIG11_READBACK[1]": "scorer read-back\nnot available",
    "FIG11_READBACK[2]": "scorer read-back\nnot available",
})

## 10 · Fig. 2c — the fifteen separability pairs, computed

In [ ]:
# =====================================================================================
#  Fig. 2c - the fifteen Jeffries-Matusita pairs, computed from your own corpus.
#
#  Nothing was wrong with the figure: ten of the fifteen pairs were never written down
#  in any master file, so the builder drew them as "?" rather than invent them. They do
#  not need a model - JM comes from per-class RGB statistics, which the dataset carries.
#  This cell measures all fifteen, checks the five that ARE recorded against the values
#  it computes, and only then fills JM_KNOWN.
#
#  JM = 2(1 - exp(-B)), B the Bhattacharyya distance between two RGB Gaussians. Range
#  0 to 2; 1.33 is the "difficult but separable" line quoted in Section 3.3.
# =====================================================================================

def jm_from_corpus(max_tiles=150, n=192, min_px=2000):
    if not (REAL_IMGS and REAL_MASKS):
        return {}, {}
    stems = [stem(p) for p in REAL_MASKS]
    if len(stems) > max_tiles:
        stems = [stems[i] for i in np.linspace(0, len(stems) - 1, max_tiles).astype(int)]
    S1 = {c: np.zeros(3) for c in CLASSES}          # sum
    S2 = {c: np.zeros((3, 3)) for c in CLASSES}     # sum of outer products
    N = {c: 0 for c in CLASSES}
    used = 0
    for s in stems:
        ip, mp = by_stem(REAL_IMGS, s), by_stem(REAL_MASKS, s)
        if not ip or not mp:
            continue
        img = thumb(ip, n)
        idx, cov = mask_to_index(thumb(mp, n))
        if img is None or idx is None or cov < 0.2:
            continue
        if img.shape[:2] != idx.shape:
            continue
        used += 1
        x = img.reshape(-1, 3).astype(np.float64)
        lab = idx.reshape(-1)
        for k, c in enumerate(CLASSES):
            v = x[lab == k]
            if len(v):
                S1[c] += v.sum(0); S2[c] += v.T @ v; N[c] += len(v)
    stats = {}
    for c in CLASSES:
        if N[c] >= min_px:
            m = S1[c] / N[c]
            cov = S2[c] / N[c] - np.outer(m, m) + np.eye(3) * 1e-6
            stats[c] = (m, cov, N[c])
    out = {}
    for i, a in enumerate(CLASSES):
        for b in CLASSES[i + 1:]:
            if a not in stats or b not in stats:
                continue
            (m1, c1, _), (m2, c2, _) = stats[a], stats[b]
            cm = 0.5 * (c1 + c2)
            d = (m1 - m2).reshape(3, 1)
            try:
                term1 = float((0.125 * d.T @ np.linalg.inv(cm) @ d).ravel()[0])
                sign, logdet = np.linalg.slogdet(cm)
                _, ld1 = np.linalg.slogdet(c1)
                _, ld2 = np.linalg.slogdet(c2)
                term2 = 0.5 * (logdet - 0.5 * (ld1 + ld2))
                B = term1 + max(term2, 0.0)
                out[frozenset([a, b])] = float(2.0 * (1.0 - np.exp(-B)))
            except np.linalg.LinAlgError:
                continue
    return out, dict(tiles=used, px={c: N[c] for c in CLASSES})


JM_COMPUTED, JM_META = ({}, {})
_want = (JM_FROM_DATA is True) or (JM_FROM_DATA == "auto" and REAL_IMGS and REAL_MASKS)
if _want:
    print("computing Jeffries-Matusita over the corpus ...")
    JM_COMPUTED, JM_META = jm_from_corpus(JM_MAX_TILES)

if JM_COMPUTED:
    print("tiles used:", JM_META["tiles"], " pixels per class:",
          ", ".join("%s %d" % (c[:4], JM_META["px"][c]) for c in CLASSES))
    print("\n   %-30s %9s %9s" % ("pair", "recorded", "computed"))
    agree = []
    for pair, rec in JM_KNOWN.items():
        got = JM_COMPUTED.get(pair)
        a, b = tuple(pair)
        print("   %-30s %9.3f %9s" % (a + " / " + b, rec,
                                      ("%.3f" % got) if got is not None else "-"))
        if got is not None:
            agree.append(abs(got - rec))
    if agree:
        print("   mean absolute difference on the five recorded pairs: %.3f" % np.mean(agree))
        if np.mean(agree) > 0.15:
            print("   NOTE: the computed values differ from the recorded ones by more than 0.15.")
            print("   Same statistic, different preprocessing - most likely the original was")
            print("   computed on the 512 cache or on a per-tile sample rather than pooled.")
            print("   Say in Section 3.3 which one the figure reports, and keep it consistent.")
    missing_before = 15 - len(JM_KNOWN)
    JM_KNOWN = dict(JM_COMPUTED)
    print("\n   Fig. 2c now has all %d pairs (%d of them previously unmeasured)."
          % (len(JM_KNOWN), missing_before))
    print("   The '?' cells and the note under the matrix disappear on their own - the")
    print("   builder draws them only when a value is missing.")
elif _want:
    print("could not compute JM (no usable image/mask pairs) - the recorded five are kept")
else:
    print("JM_FROM_DATA is off - the five recorded pairs are kept and the rest stay '?'")

## 11 · The figure builders

Three cells, one per group. They define the figures; nothing is drawn until the
selector calls them.

In [ ]:
# ====================================================================================
#  GROUP 1  ·  PLOTTED FIGURES
#  Byte-for-byte the original code, indented into functions. Nothing changed.
# ====================================================================================


# ------------------------------------------------------------------------------------
# Fig. 2 - corpus difficulty  |  figure*, 190 x 62 mm  |  S3.3, immediately after Table 1
# ------------------------------------------------------------------------------------
def _build_fig02_corpus_difficulty():
    fig, (axa, axb, axc) = plt.subplots(1, 3, figsize=(W2, 62 * MM), layout="constrained",
                                        gridspec_kw=dict(width_ratios=[1.0, 0.72, 1.05]))

    order = sorted(CLASSES, key=lambda c: SHARE[c])
    y = np.arange(len(order))

    axa.barh(y, [SHARE[c] for c in order], color=[CLR[c] for c in order], height=0.6,
             ec="white", lw=0.5)
    axa.set_xscale("log"); axa.set_xlim(0.35, 300)
    axa.set_xticks([1, 10, 100]); axa.set_xticklabels(["1", "10", "100"])
    axa.set_yticks(y); axa.set_yticklabels(order)
    axa.set_xlabel("share of annotated pixels (%)")
    for i, c in enumerate(order):
        axa.text(SHARE[c] * 1.25, i, "%.2f" % SHARE[c], va="center", fontsize=6.5)
    axa.grid(axis="x", ls=":", lw=0.4); axa.set_axisbelow(True)

    axb.barh(y, [TILES[c] for c in order], color=[CLR[c] for c in order], height=0.6,
             ec="white", lw=0.5)
    axb.set_yticks(y); axb.set_yticklabels([])
    axb.set_xlim(0, 470); axb.set_xticks([0, 200, 400])
    axb.set_xlabel("tiles containing the class (of 408)")
    for i, c in enumerate(order):
        axb.text(TILES[c] + 14, i, str(TILES[c]), va="center", fontsize=6.5)
    axb.grid(axis="x", ls=":", lw=0.4); axb.set_axisbelow(True)

    n = len(CLASSES)
    M = np.full((n, n), np.nan)
    for i in range(n):
        for j in range(i):
            v = JM_KNOWN.get(frozenset([CLASSES[i], CLASSES[j]]))
            if v is not None:
                M[i, j] = v
    cmap = plt.get_cmap("cividis").copy(); cmap.set_bad("0.90")
    im = axc.imshow(np.ma.masked_invalid(M), cmap=cmap, vmin=0, vmax=2.0)
    for i in range(n):
        for j in range(i):
            if np.isnan(M[i, j]):
                axc.text(j, i, "?", ha="center", va="center", fontsize=6.5, color="0.55")
            else:
                axc.text(j, i, ("%.3f" % M[i, j]).lstrip("0"), ha="center", va="center",
                         fontsize=6.2, color=("white" if M[i, j] < 1.1 else "0.12"))
    axc.set_xticks(range(n - 1)); axc.set_xticklabels(CLASSES[:-1], rotation=40, ha="right")
    axc.set_yticks(range(1, n)); axc.set_yticklabels(CLASSES[1:])
    axc.set_xlim(-0.5, n - 1.5); axc.set_ylim(n - 0.5, 0.5)
    axc.tick_params(length=0)
    for sp in axc.spines.values():
        sp.set_visible(False)
    cb = fig.colorbar(im, ax=axc, fraction=0.045, pad=0.03)
    cb.set_label("Jeffries-Matusita distance"); cb.ax.tick_params(labelsize=6.5)
    cb.outline.set_linewidth(0.5)
    if np.isnan(M[np.tril_indices(n, -1)]).any():
        axc.text(3.6, 1.4, "? not yet measured", ha="center", **NOTE)

    finish(fig, "fig02_corpus_difficulty", "Fig. 2", "Section 3.3",
           "immediately after Table 1, before the paragraph beginning 'Two measurements explain most "
           "of what follows'", "figure*",
           "Properties of the corpus that make it hard. (a) Share of annotated pixels by class on a "
           "logarithmic axis; the imbalance ratio between the largest and smallest class is 94:1 and "
           "the effective number of classes, exp(H), is 3.42 of 6. (b) Number of tiles containing each "
           "class, of 408; river is absent from four fifths of the corpus. (c) Jeffries-Matusita "
           "separability between every class pair on a 0-2 scale, computed from per-class Gaussians in "
           "RGB. No pair exceeds 1.33 and cultivated ground against vacant land reaches 0.051.",
           axes=[axa, axb, axc])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 3 - partitioning regimes  |  figure, 90 x 72 mm  |  S3.4, after the fold-size sentence
# ------------------------------------------------------------------------------------
def _build_fig03_partitioning_regimes():
    fig, axd = plt.subplot_mosaic([["s1", "s1"], ["s2", "s2"], ["b1", "b2"]],
                                  figsize=(W1, 72 * MM), height_ratios=[0.8, 0.8, 2.5],
                                  layout="constrained")
    ax1, ax2, ax3, ax4 = axd["s1"], axd["s2"], axd["b1"], axd["b2"]

    FOLD, BUF = 1, 2
    val_lo, val_hi = FOLD * 136, (FOLD + 1) * 136
    cmapS = ListedColormap(["#DCDCDC", "#B2182B", "#F4C6C1"])

    blk = np.zeros(N_TILES, int)
    blk[val_lo:val_hi] = 1
    blk[val_lo - BUF:val_lo] = 2
    blk[val_hi:val_hi + BUF] = 2
    ax1.imshow(blk[None, :], aspect="auto", cmap=cmapS, vmin=0, vmax=2,
               interpolation="nearest", extent=[1, N_TILES, 0, 1])

    rng = np.random.default_rng(42)
    rnd = np.zeros(N_TILES, int)
    rnd[rng.choice(N_TILES, 136, replace=False)] = 1
    ax2.imshow(rnd[None, :], aspect="auto", cmap=cmapS, vmin=0, vmax=2,
               interpolation="nearest", extent=[1, N_TILES, 0, 1])

    for ax in (ax1, ax2):
        ax.set_yticks([])
        ax.set_xticks([1, 100, 200, 300, 408])
        ax.tick_params(length=1.5, pad=1.5)
        for sp in ax.spines.values():
            sp.set_linewidth(0.5)
    ax1.set_xticklabels([])
    ax2.set_xlabel("export sequence index")
    for b in (val_lo, val_hi):
        ax1.plot([b], [1.0], marker="v", ms=3.0, color="#B2182B", clip_on=False, zorder=5)

    x = np.arange(3)
    ax3.bar(x, VAL_RESID_SHARE, color=CLR["Residential"], width=0.6, ec="white", lw=0.4)
    ax3.set_ylim(0, 96); ax3.set_yticks([0, 25, 50, 75])
    ax3.set_ylabel("validation share (%)")
    for i, v in enumerate(VAL_RESID_SHARE):
        ax3.text(i, v + 3.5, "%.1f" % v, ha="center", fontsize=6.5)

    ax4.bar(x, VAL_RIVER_SHARE, color=CLR["River"], width=0.6, ec="white", lw=0.4)
    ax4.set_ylim(0, 1.28); ax4.set_yticks([0, 0.5, 1.0])
    for i, v in enumerate(VAL_RIVER_SHARE):
        ax4.text(i, v + 0.05, "%.3f" % v, ha="center", fontsize=6.5)

    for ax in (ax3, ax4):
        ax.set_xticks(x); ax.set_xticklabels(["fold 0", "fold 1", "fold 2"])
        ax.grid(axis="y", ls=":", lw=0.4); ax.set_axisbelow(True)

    top_legend(fig, [Patch(fc="#DCDCDC", ec="0.45", lw=0.4),
                     Patch(fc="#B2182B", ec="0.45", lw=0.4),
                     Patch(fc="#F4C6C1", ec="0.45", lw=0.4)],
               ["train", "validation", "buffer"], ncol=3)

    finish(fig, "fig03_partitioning_regimes", "Fig. 3", "Section 3.4",
           "after 'Three folds of each mode give 268-272 training and 136 validation scenes'", "figure",
           "The two partitioning regimes over the 408-scene export sequence. (a) Sequence-block fold 1; "
           "the two-tile buffer dropped from the training side of each block boundary is marked by "
           "triangles and is too narrow to resolve at this scale. (b) The random control over the same "
           "scenes, in which most validation scenes have training neighbours on both sides. "
           "(c) Residential and (d) river share of the validation set in each block fold. The three "
           "block folds hold out different parts of the valley, and fold 0 contains too little river "
           "to measure it.",
           axes=[ax1, ax2, ax3, ax4], hspace=0.34)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 8 - partition effect  |  figure*, 190 x 62 mm  |  S6.1, after Table 2
# ------------------------------------------------------------------------------------
def _build_fig08_partition_effect():
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(W2, 62 * MM), layout="constrained",
                                   gridspec_kw=dict(width_ratios=[1.9, 1.0]))

    y = np.arange(len(ARCHS))[::-1]
    for k, a in enumerate(ARCHS):
        b, r = np.array(MIOU_BLOCK[a]), np.array(MIOU_RANDOM[a])
        axa.plot([b.mean(), r.mean()], [y[k], y[k]], color="0.78", lw=1.0, zorder=1)
        axa.plot(b, [y[k] + 0.19] * 3, "o", ms=3.0, mfc="none", mec=C_BLOCK, mew=0.8, zorder=2)
        axa.plot(r, [y[k] - 0.19] * 3, "s", ms=2.8, mfc="none", mec=C_RANDOM, mew=0.8, zorder=2)
        axa.plot(b.mean(), y[k], "o", ms=5.0, color=C_BLOCK, mec="white", mew=0.6, zorder=4)
        axa.plot(r.mean(), y[k], "s", ms=4.6, color=C_RANDOM, mec="white", mew=0.6, zorder=4)
    axa.set_yticks(y); axa.set_yticklabels(ARCHS)
    axa.set_xlabel("mean intersection over union")
    axa.set_xlim(0.25, 0.62); axa.set_ylim(-0.6, len(ARCHS) - 0.4)
    axa.grid(axis="x", ls=":", lw=0.4); axa.set_axisbelow(True)

    axb.barh(y, [SD_RATIO[a] for a in ARCHS], color=[ACLR[a] for a in ARCHS], height=0.5,
             ec="white", lw=0.5)
    axb.axvline(1, color="0.45", lw=0.6, ls="--", zorder=1)
    for k, a in enumerate(ARCHS):
        axb.text(SD_RATIO[a] + 0.6, y[k], "%.1f" % SD_RATIO[a], va="center", fontsize=6.5)
    axb.set_yticks(y); axb.set_yticklabels([])
    axb.set_xlim(0, 24); axb.set_ylim(-0.6, len(ARCHS) - 0.4)
    axb.set_xlabel("block SD / random SD")
    axb.grid(axis="x", ls=":", lw=0.4); axb.set_axisbelow(True)

    top_legend(fig, [Line2D([], [], marker="o", ls="none", mfc="none", mec=C_BLOCK),
                     Line2D([], [], marker="o", ls="none", color=C_BLOCK),
                     Line2D([], [], marker="s", ls="none", mfc="none", mec=C_RANDOM),
                     Line2D([], [], marker="s", ls="none", color=C_RANDOM)],
               ["sequence-block, per fold", "sequence-block, mean",
                "random, per fold", "random, mean"], ncol=4)

    finish(fig, "fig08_partition_effect", "Fig. 8", "Section 6.1",
           "immediately after Table 2, before 'The two right-hand columns carry most of the study's "
           "methodological weight'", "figure*",
           "The effect of the partitioning protocol. (a) Mean intersection over union under both "
           "regimes; open markers are the three folds, filled markers the regime mean, and the "
           "connecting line is the gap, which runs in the same direction for every architecture and "
           "spans 0.051 to 0.095. The three lowest block-split points, one per architecture, are all "
           "fold 0. (b) Ratio of the between-fold standard deviation under blocking to that under "
           "random splitting; the dashed line marks equal spread.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 14 - proposed model against its backbone  |  figure, 90 x 52 mm  |  S6.1, after the six-fold-run sentence
# ------------------------------------------------------------------------------------
def _build_fig14_proposed_vs_backbone():
    deltas, cols = [], []
    for d, c in ((MIOU_BLOCK, C_BLOCK), (MIOU_RANDOM, C_RANDOM)):
        for f in range(3):
            deltas.append(d["TC-SegFormer"][f] - d["SegFormer-B0"][f]); cols.append(c)

    fig, ax = plt.subplots(figsize=(W1, 52 * MM), layout="constrained")
    x = np.arange(6)
    ax.bar(x, deltas, color=cols, width=0.6, ec="white", lw=0.5)
    for i, v in enumerate(deltas):
        ax.text(i, v + 0.0038, "%.4f" % v, ha="center", fontsize=6.5)
    ax.axhline(0, color="0.25", lw=0.7)
    ax.axvline(2.5, color="0.85", lw=0.6)
    ax.set_xticks(x); ax.set_xticklabels(["fold 0", "fold 1", "fold 2"] * 2)
    ax.set_ylim(0, 0.132)
    ax.set_ylabel("mIoU, proposed minus backbone")
    ax.grid(axis="y", ls=":", lw=0.4); ax.set_axisbelow(True)

    top_legend(fig, [Patch(fc=C_BLOCK), Patch(fc=C_RANDOM)],
               ["sequence-block split", "random split"], ncol=2)

    finish(fig, "fig14_proposed_vs_backbone", "Fig. 14", "Section 6.1",
           "after 'or 11.7 % to 24.0 % in relative terms, for 0.078 M added parameters'", "figure",
           "The proposed model minus its own SegFormer-B0 backbone across all six fold-runs. The two "
           "share an encoder, an initialisation and a schedule, so this is the study's only comparison "
           "in which architecture is not confounded with capacity or pretraining. The proposed model "
           "wins every fold-run for 1.9 % more parameters.", axes=None)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 15 - per-class IoU and spatial penalty  |  figure*, 190 x 56 mm  |  S6.2, in place of Table 3
# ------------------------------------------------------------------------------------
def _build_fig15_perclass_iou_penalty():
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(W2, 56 * MM), layout="constrained")

    B = np.array([PC_BLOCK[a] for a in ARCHS])
    P = np.array([PC_RANDOM[a] for a in ARCHS]) - B

    im1 = axa.imshow(B, cmap="viridis", vmin=0.05, vmax=0.85, aspect="auto")
    im2 = axb.imshow(P, cmap="RdBu_r", norm=TwoSlopeNorm(vmin=-0.02, vcenter=0.0, vmax=0.16),
                     aspect="auto")
    for ax, D, fmtstr, thr in ((axa, B, "%.3f", 0.55), (axb, P, "%+.3f", 0.105)):
        for i in range(len(ARCHS)):
            for j in range(len(CLASSES)):
                ax.text(j, i, fmtstr % D[i, j], ha="center", va="center", fontsize=6.2,
                        color=("white" if (D[i, j] < thr if ax is axa else D[i, j] > thr) else "0.12"))
        ax.set_xticks(range(len(CLASSES)))
        ax.set_xticklabels(CLASSES, rotation=40, ha="right")
        ax.set_yticks(range(len(ARCHS)))
        ax.tick_params(length=0)
    axa.set_yticklabels(ARCHS); axb.set_yticklabels([])
    for im, ax, lab in ((im1, axa, "intersection over union"),
                        (im2, axb, "random minus block")):
        cb = fig.colorbar(im, ax=ax, fraction=0.042, pad=0.02)
        cb.set_label(lab); cb.ax.tick_params(labelsize=6.5); cb.outline.set_linewidth(0.5)

    finish(fig, "fig15_perclass_iou_penalty", "Fig. 15", "Section 6.2",
           "in place of Table 3, after the paragraph ending 'nothing architectural moves it'", "figure*",
           "Per-class results by architecture. (a) Intersection over union averaged over the three "
           "sequence-block folds. (b) The spatial penalty, random minus block, for the same cells. The "
           "penalty is not spread evenly: it concentrates in river and vacant land, which lose 0.143 "
           "and 0.130 on average, against 0.014 for cultivated ground.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 16 - LoveDA per-class  |  figure, 90 x 58 mm  |  S6.3, after the sentence on barren
# ------------------------------------------------------------------------------------
def _build_fig16_loveda_perclass():
    fig, ax = plt.subplots(figsize=(W1, 58 * MM), layout="constrained")
    y = np.arange(len(LOVEDA_CLASSES))[::-1]
    h = 0.36
    ax.barh(y + h / 2, LOVEDA_B0, height=h, color="#2C7FB8", ec="white", lw=0.4)
    ax.barh(y - h / 2, LOVEDA_TC, height=h, color="#B2182B", ec="white", lw=0.4)
    LX = 0.70
    for k in range(len(LOVEDA_CLASSES)):
        d = LOVEDA_TC[k] - LOVEDA_B0[k]
        ax.text(LX, y[k], "%+.4f" % d, va="center", ha="left", fontsize=6.5,
                color=("#B2182B" if d < 0 else "#1A7A3C"))
    ax.text(LX, len(LOVEDA_CLASSES) - 0.55, "TC - B0", va="center", ha="left", fontsize=6.5,
            color="0.35")
    ax.set_yticks(y); ax.set_yticklabels(LOVEDA_CLASSES)
    ax.set_xlabel("intersection over union")
    ax.set_xlim(0, 0.86); ax.set_xticks([0, 0.2, 0.4, 0.6])
    ax.set_ylim(-0.6, len(LOVEDA_CLASSES) - 0.25)
    ax.grid(axis="x", ls=":", lw=0.4); ax.set_axisbelow(True)

    top_legend(fig, [Patch(fc="#2C7FB8"), Patch(fc="#B2182B")],
               ["SegFormer-B0", "proposed model"], ncol=2)

    finish(fig, "fig16_loveda_perclass", "Fig. 16", "Section 6.3",
           "after 'and it is the class this architecture handles worst on both datasets'", "figure",
           "Per-class intersection over union on the LoveDA validation split under a matched 20,000-step "
           "budget. The mean is a tie, 0.4730 against 0.4718, but it conceals a structured "
           "redistribution: the proposed model gains on water, building, agriculture and road, and "
           "loses on barren, forest and background. Water and road are the two classes to which the "
           "topology term is applied in this configuration; barren is LoveDA's structural analogue of "
           "vacant land.", axes=None)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 10 - generator checkpoint selection  |  figure, 90 x 58 mm  |  S6.4, checkpoint-selection paragraph
# ------------------------------------------------------------------------------------
def _build_fig10_generator_selection():
    fig, ax = plt.subplots(figsize=(W1, 58 * MM), layout="constrained")
    ax2 = ax.twinx()
    ax2.spines["right"].set_visible(True); ax2.spines["top"].set_visible(False)

    ax.plot(S2_STEP, S2_LAYOUT, "-o", color="#1A7A3C", ms=2.8, lw=1.1)
    ax.set_xlabel("training step")
    ax.set_ylabel("layout fidelity (mIoU)", color="#1A7A3C")
    ax.tick_params(axis="y", colors="#1A7A3C"); ax.spines["left"].set_color("#1A7A3C")
    ax.set_ylim(0.20, 0.47); ax.set_xlim(0, 7700)

    partial = len(S2_EPS_STEP) <= 3
    ax2.plot(S2_EPS_STEP, S2_EPS_MSE, ("--o" if partial else "-o"), color="#B2182B", ms=2.8, lw=1.0)
    ax2.set_ylabel("validation " + r"$\epsilon$" + "-MSE", color="#B2182B")
    ax2.tick_params(axis="y", colors="#B2182B"); ax2.spines["right"].set_color("#B2182B")
    ax2.set_ylim(0.1148, 0.1205)

    ax.plot(S2_STEP[-1], S2_LAYOUT[-1], "o", ms=7, mfc="none", mec="#1A7A3C", mew=1.2)
    ax.plot(S2_STEP[0], S2_LAYOUT[0], "s", ms=7, mfc="none", mec="#B2182B", mew=1.2)
    ax.grid(axis="y", ls=":", lw=0.4); ax.set_axisbelow(True)
    if partial:
        ax.text(3850, 0.222, "denoising trace: logged endpoints only", ha="center", **NOTE)

    top_legend(fig, [Line2D([], [], color="#1A7A3C", marker="o", ms=2.8),
                     Line2D([], [], color="#B2182B", marker="o", ms=2.8, ls="--"),
                     Line2D([], [], color="#1A7A3C", marker="o", ms=6, mfc="none", ls="none"),
                     Line2D([], [], color="#B2182B", marker="s", ms=6, mfc="none", ls="none")],
               ["layout fidelity", "denoising error", "selected checkpoint",
                "denoising-selected"], ncol=2)

    finish(fig, "fig10_generator_selection", "Fig. 10", "Section 6.4",
           "in the paragraph beginning 'Two findings here are methodological rather than numerical', "
           "after the sentence quoting 0.1158 and 0.1186", "figure",
           "Selecting the image stage on its denoising objective selects the wrong checkpoint. Over the "
           "run, validation denoising error rises while layout fidelity climbs from 0.24 to 0.42; the "
           "two criteria move in opposite senses of 'better'. Selecting on fidelity gives a generator "
           "at 0.946 of the real-image ceiling, selecting on denoising error one at 0.651.", axes=None)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 20 - synthetic pool quality  |  figure, 90 x 66 mm  |  S6.4, after the memorisation sentence
# ------------------------------------------------------------------------------------
def _build_fig20_synthetic_pool_quality():
    fig, (axa, axb) = plt.subplots(2, 1, figsize=(W1, 66 * MM), layout="constrained")

    x = np.arange(3)
    axa.bar(x, S3_LAYOUT, width=0.5, color="#2C7FB8", ec="white", lw=0.5)
    axa.axhline(S3_CEIL, color="#B2182B", lw=0.9, ls="--")
    for i, v in enumerate(S3_LAYOUT):
        axa.text(i, v / 2, "%.3f" % v, ha="center", va="center", fontsize=6.5, color="white")
    axa.set_xticks(x); axa.set_xticklabels(S3_SETS)
    axa.set_ylabel("layout mIoU"); axa.set_ylim(0, 0.60); axa.set_yticks([0, 0.2, 0.4])
    axa.grid(axis="y", ls=":", lw=0.4); axa.set_axisbelow(True)

    spaces = list(S3_MEM.keys()); w = 0.24
    for k, s in enumerate(S3_SETS):
        axb.bar(np.arange(len(spaces)) + (k - 1) * w, [S3_MEM[sp][k] for sp in spaces],
                width=w, color=S3_COL[k], ec="white", lw=0.4)
    axb.axhline(1.0, color="#B2182B", lw=0.9, ls="--")
    axb.set_xticks(range(len(spaces))); axb.set_xticklabels(spaces)
    axb.set_ylabel("NN distance ratio"); axb.set_ylim(0, 1.62); axb.set_yticks([0, 0.5, 1.0, 1.5])
    axb.grid(axis="y", ls=":", lw=0.4); axb.set_axisbelow(True)

    top_legend(fig, [Patch(fc="#2C7FB8"), Line2D([], [], color="#B2182B", ls="--")] +
               [Patch(fc=c) for c in S3_COL],
               ["synthetic pool", "real reference", "R1", "R2", "R3"], ncol=5)

    finish(fig, "fig20_synthetic_pool_quality", "Fig. 20", "Section 6.4",
           "after 'only raw pixel space falls below one, at 0.89-0.90'", "figure",
           "Quality of the synthetic pool at three set sizes. (a) Layout fidelity, the intersection "
           "over union a frozen scorer achieves on a generated image against the mask that generated "
           "it; the dashed line is the same scorer's ceiling of 0.4646 on real imagery, so the three "
           "pools sit at 0.933 to 0.946 of it. (b) Nearest-neighbour retrieval distance from a "
           "synthetic sample to the training set, as a ratio to the same statistic for a held-out real "
           "tile (dashed line). Feature and mask space are at or above one; the sub-unit pixel-space "
           "ratio is the expected consequence of a generator matching the dataset's colour statistics.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 12 - the downstream result  |  figure*, 190 x 58 mm  |  S6.5, after Table 4  (part-2 headline)
# ------------------------------------------------------------------------------------
def _build_fig12_downstream_forest_plot():
    fig, axes3 = plt.subplots(1, 3, figsize=(W2, 58 * MM), sharey=True, layout="constrained")
    spec = [(axes3[0], S4_MIOU, S4_MEAN["mIoU"], (-0.058, 0.026), "mean IoU"),
            (axes3[1], S4_RIVER, S4_MEAN["River"], (-0.150, 0.070), "river IoU"),
            (axes3[2], S4_ROAD, S4_MEAN["Road"], (-0.100, 0.042), "road IoU")]

    y = np.arange(len(ARCHS))[::-1]
    for ax, D, mean, xlim, lab in spec:
        ax.axvline(0, color="0.25", lw=0.8, zorder=1)
        ax.axvline(mean, color="0.65", lw=0.7, ls="--", zorder=1)
        for k, a in enumerate(ARCHS):
            d, lo, hi, sig = D[a]
            ax.plot([lo, hi], [y[k], y[k]], color=ACLR[a], lw=1.2, zorder=2)
            for e in (lo, hi):
                ax.plot([e, e], [y[k] - 0.12, y[k] + 0.12], color=ACLR[a], lw=0.9, zorder=2)
            ax.plot(d, y[k], "o", ms=4.6, color=(ACLR[a] if sig else "white"), mec=ACLR[a],
                    mew=1.0, zorder=3)
        ax.set_xlim(*xlim); ax.set_ylim(-0.55, len(ARCHS) - 0.45)
        ax.set_xlabel("change in " + lab + " (R1 - R0)")
        ax.xaxis.set_major_locator(mpl.ticker.MaxNLocator(4))
        ax.grid(axis="x", ls=":", lw=0.4); ax.set_axisbelow(True)
    axes3[0].set_yticks(y); axes3[0].set_yticklabels(ARCHS)

    top_legend(fig, [Line2D([], [], marker="o", ls="none", color="0.35"),
                     Line2D([], [], marker="o", ls="none", mfc="white", mec="0.35"),
                     Line2D([], [], color="0.65", ls="--")],
               ["95 % interval excludes zero", "not resolvable", "mean of five"], ncol=3)

    finish(fig, "fig12_downstream_forest_plot", "Fig. 12", "Section 6.5",
           "immediately after Table 4, before the paragraph beginning 'The per-class picture is where "
           "the experiment stops being a null'", "figure*",
           "Effect of adding 804 synthetic image-label pairs to 804 real ones at a matched 6000-step "
           "budget, on block fold 1. Change in (a) mean, (b) river and (c) road intersection over "
           "union, with 95 % paired-bootstrap intervals over the 136 real validation scenes. Every "
           "architecture loses on all three measures. Means over the five are -0.0140, -0.0570 and "
           "-0.0310. Filled markers mark deltas whose interval excludes zero.",
           axes=list(axes3))
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 17 - per-class damage  |  figure*, 190 x 54 mm  |  S6.5, in place of the left half of Table 5
# ------------------------------------------------------------------------------------
def _build_fig17_perclass_downstream_delta():
    D = np.array([S4_DELTA[a] for a in ARCHS])
    S = np.array([S4_SIG[a] for a in ARCHS])
    Dm = np.vstack([D, np.array(S4_DELTA_MEAN)])
    rows = ARCHS + ["mean of five"]

    fig, ax = plt.subplots(figsize=(W2, 54 * MM), layout="constrained")
    lim = np.abs(Dm).max()
    im = ax.imshow(Dm, cmap="RdBu", norm=TwoSlopeNorm(vmin=-lim, vcenter=0, vmax=lim), aspect="auto")
    for i in range(Dm.shape[0]):
        for j in range(Dm.shape[1]):
            v = Dm[i, j]
            star = "*" if (i < len(ARCHS) and S[i, j]) else ""
            ax.text(j, i, "%+.4f" % v + star, ha="center", va="center", fontsize=6.3,
                    color=("white" if abs(v) > lim * 0.55 else "0.12"))
    ax.set_xticks(range(len(S4_CLASS_ORDER))); ax.set_xticklabels(S4_CLASS_ORDER)
    ax.set_yticks(range(len(rows))); ax.set_yticklabels(rows)
    ax.tick_params(length=0)
    ax.axhline(len(ARCHS) - 0.5, color="0.3", lw=0.8)
    ax.axvline(1.5, color="0.3", lw=0.8)
    cb = fig.colorbar(im, ax=ax, fraction=0.022, pad=0.012)
    cb.set_label(r"$\Delta$ IoU (R1 - R0)"); cb.ax.tick_params(labelsize=6.5)
    cb.outline.set_linewidth(0.5)

    finish(fig, "fig17_perclass_downstream_delta", "Fig. 17", "Section 6.5",
           "in place of the left half of Table 5, after 'The per-class picture is where the experiment "
           "stops being a null and becomes a diagnosis'", "figure*",
           "Per-class change in intersection over union from adding the synthetic pool, by "
           "architecture. An asterisk marks a cell whose 95 % paired-bootstrap interval excludes zero. "
           "The two classes left of the rule, river and road, are the pre-registered targets of the "
           "generative pipeline and are the two it damaged most, in every architecture. The only "
           "classes that gain on average are vacant land and cultivated ground, neither of them a thin "
           "structure.", axes=None)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 18 - what predicts the damage  |  figure*, 140 x 58 mm  |  S7.3, after the ratio-versus-absolute sentence
# ------------------------------------------------------------------------------------
def _build_fig18_mechanism_fidelity_ratio():
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(W15, 58 * MM), sharey=True, layout="constrained")

    names = list(FIDELITY.keys())
    dy = [FIDELITY[c][0] for c in names]
    syn = [FIDELITY[c][1] for c in names]
    ratio = [FIDELITY[c][2] for c in names]

    OFF_A = {"River": (0.040, 0, "left", "center"), "Road": (0.040, 0, "left", "center"),
             "Forest": (-0.030, 0, "right", "center"), "Residential": (0, -0.0055, "center", "top"),
             "Agricultural": (0, 0.0050, "center", "bottom"),
             "UnusedLand": (0, 0.0050, "center", "bottom")}
    OFF_B = {"River": (0.032, 0, "left", "center"), "Road": (0.032, 0, "left", "center"),
             "UnusedLand": (0, 0.0050, "center", "bottom"),
             "Agricultural": (0, 0.0050, "center", "bottom"),
             "Forest": (0, 0.0050, "center", "bottom"),
             "Residential": (0, -0.0055, "center", "top")}

    for ax, xs, lab, rho, OFF, xlim in (
            (axa, ratio, "layout fidelity relative to the real-image ceiling", "+0.83", OFF_A, (0.32, 1.42)),
            (axb, syn, "absolute layout IoU on synthetic imagery", "+0.43", OFF_B, (-0.03, 0.99))):
        ax.axhline(0, color="0.25", lw=0.7, zorder=1)
        for k, c in enumerate(names):
            ax.plot(xs[k], dy[k], "o", ms=5.8, color=CLR[c], mec="white", mew=0.6, zorder=3)
            ox, oy, ha, va = OFF[c]
            ax.text(xs[k] + ox, dy[k] + oy, c, fontsize=6.5, color="0.2", ha=ha, va=va)
        ax.set_xlabel(lab); ax.set_xlim(*xlim)
        ax.grid(ls=":", lw=0.4); ax.set_axisbelow(True)
        ax.text(0.97, 0.05, "Spearman " + r"$\rho$" + " = " + rho, transform=ax.transAxes,
                ha="right", va="bottom", **NOTE)
    axa.set_ylabel(r"mean $\Delta$ IoU from the synthetic pool")
    axa.set_ylim(-0.078, 0.032)

    finish(fig, "fig18_mechanism_fidelity_ratio", "Fig. 18", "Section 7.3",
           "after 'what predicts damage is not how badly the generator renders a class, but how badly "
           "it renders it relative to what that class could achieve on real imagery anyway'", "figure*",
           "What predicts the downstream damage. (a) Mean change in intersection over union against "
           "layout fidelity expressed as a ratio to the frozen scorer's own ceiling on real imagery. "
           "(b) The same against the absolute layout fidelity on synthetic imagery. The ratio orders "
           "the classes; the absolute value does not. Vacant land carries 35 % contamination and still "
           "gains, because it has almost no texture identity to corrupt. With six non-independent "
           "classes the correlation coefficients are descriptive rather than inferential.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 13 - component ablation  |  figure*, 140 x 62 mm  |  S7.2, after Table 6
# ------------------------------------------------------------------------------------
def _build_fig13_component_ablation():
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(W15, 62 * MM), layout="constrained",
                                   gridspec_kw=dict(width_ratios=[1.0, 1.25]))

    x = np.arange(len(ABL_ORDER))
    axa.bar(x, [ABL_MIOU[k] for k in ABL_ORDER], color=[ABL_COL[k] for k in ABL_ORDER],
            width=0.62, ec="white", lw=0.5)
    for i, k in enumerate(ABL_ORDER):
        axa.text(i, ABL_MIOU[k] + 0.004, "%.4f" % ABL_MIOU[k], ha="center", fontsize=6.2)
    axa.set_xticks(x)
    axa.set_xticklabels([k.replace("- ", "\u2212") for k in ABL_ORDER], rotation=40, ha="right")
    axa.set_ylabel("mIoU, block fold 1")
    axa.set_ylim(0.44, 0.585); axa.set_yticks([0.45, 0.50, 0.55])
    axa.grid(axis="y", ls=":", lw=0.4); axa.set_axisbelow(True)

    variants = ["- detail path", "- sampler", "- soft-clDice", "- CSA"]
    y = np.arange(len(CLASSES))[::-1]
    h = 0.19
    axb.axvspan(-DEFENSIBLE, DEFENSIBLE, color="0.88", zorder=0)
    for k, v in enumerate(variants):
        axb.barh(y + (1.5 - k) * h, [ABL_PERCLASS[v][c] for c in CLASSES], height=h,
                 color=ABL_COL[v], ec="white", lw=0.3, zorder=2)
    axb.axvline(0, color="0.25", lw=0.7, zorder=3)
    axb.set_yticks(y); axb.set_yticklabels(CLASSES)
    axb.set_xlabel(r"$\Delta$ IoU when the component is removed")
    axb.set_xlim(-0.046, 0.016)
    axb.grid(axis="x", ls=":", lw=0.4); axb.set_axisbelow(True)

    top_legend(fig, [Patch(fc=ABL_COL[v]) for v in variants] + [Patch(fc="0.88")],
               [v.replace("- ", "\u2212") for v in variants] + ["not defensible at one seed"], ncol=5)

    finish(fig, "fig13_component_ablation", "Fig. 13", "Section 7.2",
           "immediately after Table 6, before 'The per-class view is more useful than the headline'",
           "figure*",
           "Leave-one-out ablation on block fold 1. (a) Mean intersection over union for the full "
           "model, each variant with one component disabled, and the bare backbone. The four "
           "leave-one-out differences sum to 0.0284 against a full-minus-backbone gap of 0.0925, so "
           "the decomposition explains under a third of the effect it is meant to explain. (b) The "
           "same differences per class, where the components separate: the detail path acts on river "
           "and road, the topology term on road and river only, and the sampler on vacant land. The "
           "shaded band is the 0.005 below which nothing is defensible at a single seed, which "
           "excludes the reweighting result entirely.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 21 - weighted loss and topology  |  figure*, 140 x 58 mm  |  S7.3, or Supplementary
# ------------------------------------------------------------------------------------
def _build_fig21_fwiou_and_cldice():
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(W15, 58 * MM), layout="constrained",
                                   gridspec_kw=dict(width_ratios=[1.0, 1.15]))

    y = np.arange(len(ARCHS))[::-1]
    h = 0.34
    axa.barh(y + h / 2, [FWIOU[a][1] for a in ARCHS], height=h, color="#B2182B", ec="white", lw=0.4)
    axa.barh(y - h / 2, [FWIOU[a][0] for a in ARCHS], height=h, color="#9ECAE1", ec="white", lw=0.4)
    axa.axvline(0, color="0.25", lw=0.7)
    axa.set_yticks(y); axa.set_yticklabels(ARCHS)
    axa.set_xlim(-0.042, 0.004); axa.set_xlabel("change from adding synthetic pairs")
    axa.grid(axis="x", ls=":", lw=0.4); axa.set_axisbelow(True)

    axb.barh(y + h / 2, [CLDICE[a][0] for a in ARCHS], height=h, color=CLR["Road"], ec="white", lw=0.4)
    axb.barh(y - h / 2, [CLDICE[a][1] for a in ARCHS], height=h, color=CLR["River"], ec="white", lw=0.4)
    axb.axvline(0, color="0.25", lw=0.7)
    axb.set_yticks(y); axb.set_yticklabels([])
    axb.set_xlim(-0.100, 0.010); axb.set_xlabel(r"$\Delta$ centreline Dice")
    axb.grid(axis="x", ls=":", lw=0.4); axb.set_axisbelow(True)

    axr = axb.twinx()
    axr.spines["right"].set_visible(True); axr.spines["top"].set_visible(False)
    axr.set_ylim(axb.get_ylim()); axr.set_yticks(y)
    axr.set_yticklabels([str(CLDICE[a][2]) + "\u2192" + str(CLDICE[a][3]) for a in ARCHS])
    axr.set_ylabel("river images entering the mean, R0 to R1", fontsize=7)
    axr.tick_params(length=0, labelsize=6.5)

    top_legend(fig, [Patch(fc="#B2182B"), Patch(fc="#9ECAE1")],
               [r"$\Delta$ mIoU (unweighted)", r"$\Delta$ FwIoU (pixel-weighted)"],
               ncol=1, loc="outside upper left")
    top_legend(fig, [Patch(fc=CLR["Road"]), Patch(fc=CLR["River"])],
               ["road", "river"], ncol=2, loc="outside upper right")

    finish(fig, "fig21_fwiou_and_cldice", "Fig. 21", "Section 7.3",
           "after the frequency-weighted intersection over union sentence, or Supplementary", "figure*",
           "Where the loss sits, and what it does to topology. (a) Change in unweighted mean "
           "intersection over union beside change in frequency-weighted intersection over union. Both "
           "fall for every architecture, so the degradation is real; the weighted one falls by only "
           "40 % of the unweighted amount, so it is specifically a rare-class degradation. (b) Change "
           "in centreline Dice for the two thin classes. The right-hand axis gives the number of "
           "images entering each river mean in the two arms: models predict river on more images "
           "under R1, so the river delta is confounded with a change in the set averaged over.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 19 - label-representation ablation  |  figure, 90 x 52 mm  |  S4.2, or Supplementary
# ------------------------------------------------------------------------------------
def _build_fig19_label_representation():
    fig, (axa, axb) = plt.subplots(1, 2, figsize=(W1, 52 * MM), layout="constrained",
                                   gridspec_kw=dict(width_ratios=[0.85, 1.15]))

    arms = list(S1_KL.keys())
    axa.bar([0, 1], [S1_KL[a] for a in arms], color=[S1_COL[a] for a in arms], width=0.55,
            ec="white", lw=0.5)
    axa.set_yscale("log"); axa.set_ylim(3e-6, 30)
    axa.set_xticks([0, 1]); axa.set_xticklabels(["binary", "one-hot"])
    axa.set_ylabel("class-share KL")
    for i, a in enumerate(arms):
        axa.text(i, S1_KL[a] * 2.2, "%.5f" % S1_KL[a], ha="center", fontsize=6.2)
    axa.grid(axis="y", ls=":", lw=0.4); axa.set_axisbelow(True)

    w = 0.34; xs = np.arange(2)
    for i, a in enumerate(arms):
        axb.bar(xs + (i - 0.5) * w, [S1_FRAG[a]["Road"], S1_FRAG[a]["River"]], width=w,
                color=S1_COL[a], ec="white", lw=0.4)
        for j, c in enumerate(["Road", "River"]):
            axb.text(j + (i - 0.5) * w, S1_FRAG[a][c] + 0.18, "%.2f" % S1_FRAG[a][c],
                     ha="center", fontsize=6.2)
    axb.axhline(1.0, color="0.35", lw=0.8, ls="--")
    axb.set_xticks(xs); axb.set_xticklabels(["road", "river"])
    axb.set_ylabel("components, generated / real")
    axb.set_ylim(0, 8.4); axb.set_yticks([0, 2, 4, 6, 8])
    axb.grid(axis="y", ls=":", lw=0.4); axb.set_axisbelow(True)

    top_legend(fig, [Patch(fc=S1_COL[a]) for a in arms] + [Line2D([], [], color="0.35", ls="--")],
               ["binary label code", "one-hot", "real masks"], ncol=3)

    finish(fig, "fig19_label_representation", "Fig. 19", "Section 4.2",
           "after 'a useful negative result for anyone applying binary label coding to a class set "
           "whose boundaries are not uniformly distributed', or Supplementary", "figure",
           "The label-representation ablation in the mask generator. (a) Kullback-Leibler divergence "
           "between the generated and real class-share marginals, on a logarithmic axis. (b) Connected "
           "components per thin class in generated masks as a ratio to real masks; the dashed line is "
           "parity. Under a binary label code a single-bit error moves a pixel to an arbitrary other "
           "class, which shatters structures two to three pixels wide; one-hot coding gives every "
           "class an orthogonal channel and brings fragmentation back to about 1.5 times real.",
           axes=[axa, axb])
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. A.1(b) - scale recovery  |  figure, 90 x 48 mm  |  Appendix A, after Table A.1
# ------------------------------------------------------------------------------------
def _build_figA1_scale_recovery_budget():
    fig, ax = plt.subplots(figsize=(W1, 48 * MM), layout="constrained")
    names = [c[0] for c in GSD_CONTROLS]
    true_m = np.array([c[1] for c in GSD_CONTROLS])
    gsd = np.array([c[2] for c in GSD_CONTROLS])
    y = np.arange(len(names))[::-1]

    ax.axvspan(GSD_COMBINED - GSD_HALFRANGE, GSD_COMBINED + GSD_HALFRANGE,
               color="#2C7FB8", alpha=0.13, lw=0, zorder=0)
    ax.axvline(GSD_COMBINED, color="#2C7FB8", lw=1.0, zorder=1)
    for k in range(len(names)):
        ax.plot([GSD_COMBINED, gsd[k]], [y[k], y[k]], color="0.8", lw=0.7, zorder=1)
        ax.scatter(gsd[k], y[k], s=18 + 42 * (true_m[k] / true_m.max()), color="#B2182B",
                   ec="white", lw=0.6, zorder=3)
        ax.text(gsd[k], y[k] + 0.30, "%.4f" % gsd[k], ha="center", fontsize=6.3)
    ax.set_yticks(y)
    ax.set_yticklabels([n + " (" + ("%.2f" % m) + " m)" for n, m in zip(names, true_m)])
    ax.set_xlabel("ground sampling distance (m per pixel)")
    ax.set_xlim(0.2960, 0.3120); ax.set_ylim(-0.55, len(names) - 0.35)
    ax.grid(axis="x", ls=":", lw=0.4); ax.set_axisbelow(True)

    top_legend(fig, [Line2D([], [], color="#2C7FB8"), Patch(fc="#2C7FB8", alpha=0.13)],
               ["length-weighted estimate, 0.3016", "half-range, " + r"$\pm$" + "0.0049"], ncol=2)

    finish(fig, "figA1_scale_recovery_budget", "Fig. A.1", "Appendix A",
           "immediately after Table A.1", "figure",
           "Ground sampling distance from four controls whose dimensions are fixed by regulation "
           "independently of the size of the pitch on which they are marked. Marker size is "
           "proportional to baseline length. The two long baselines cluster and the two short ones "
           "scatter, which is what a constant per-line localisation error produces; no control lies "
           "more than 0.89 pixels from the combined estimate.", axes=None)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. S1 - efficiency  |  figure, 90 x 54 mm  |  Supplementary, or S6.6
# ------------------------------------------------------------------------------------
def _build_figS1_efficiency():
    fig, ax = plt.subplots(figsize=(W1, 54 * MM), layout="constrained")
    LAB = {"U-Net (R18)": (-0.5, 0, "right", "center"),
           "UNetFormer (R18)": (-0.5, 0, "right", "center"),
           "SegFormer-B0": (0.45, 0, "left", "center"),
           "SegNeXt-T": (0.45, 0.0045, "left", "bottom"),
           "TC-SegFormer": (0.45, -0.0045, "left", "top")}
    ax.annotate("", xy=(EFF["TC-SegFormer"], np.mean(MIOU_BLOCK["TC-SegFormer"])),
                xytext=(EFF["SegFormer-B0"], np.mean(MIOU_BLOCK["SegFormer-B0"])),
                arrowprops=dict(arrowstyle="->", lw=0.8, color="#B2182B"), zorder=2)
    for a in ARCHS:
        p, m = EFF[a], float(np.mean(MIOU_BLOCK[a]))
        ax.scatter(p, m, s=46, color=ACLR[a], ec="white", lw=0.7, zorder=3)
        dx, dyy, ha, va = LAB[a]
        ax.text(p + dx, m + dyy, a, fontsize=6.5, ha=ha, va=va, color="0.2")
    ax.set_xlabel("parameters (M)")
    ax.set_ylabel("mIoU, sequence-block mean")
    ax.set_xlim(0, 17.5); ax.set_ylim(0.375, 0.50)
    ax.grid(ls=":", lw=0.4); ax.set_axisbelow(True)

    finish(fig, "figS1_efficiency", "Fig. S1", "Supplementary",
           "or Section 6.6 if the efficiency discussion stays in the main text", "figure",
           "Parameter count against sequence-block mean intersection over union. The arrow marks the "
           "0.078 M parameters, 1.9 % of the backbone, that separate the proposed model from "
           "SegFormer-B0. Throughput is deliberately not plotted: because every architecture was "
           "trained twice under identical conditions, the spread between its two measurements sets an "
           "empirical repeatability floor of 12.4 %, below which no difference is real.", axes=None)
    plt.show()


print('defined: 16 figure builders')

In [ ]:
# ====================================================================================
#  GROUP 2  ·  BLOCK DIAGRAMS
#  Byte-for-byte the original code, indented into functions. Nothing changed.
# ====================================================================================


# ------------------------------------------------------------------------------------
# Fig. 4 - study pipeline  |  figure*, 190 mm  |  S4, after the opening paragraph
# ------------------------------------------------------------------------------------
def _build_fig04_study_pipeline():
    UW, UH = 748, 250
    fw = W2; fh = panel_height(fw, UW, UH)
    fig, ax = plt.subplots(figsize=(fw, fh), layout="constrained")
    diagram_axes(ax, UW, UH)

    ax.plot([374, 374], [16, 236], color="#D0D0D0", lw=0.8, ls=(0, (3, 3)), zorder=0)
    ax.text(180, 12, "segmentation half", ha="center", fontsize=7.5, color="#555555")
    ax.text(560, 12, "generative half", ha="center", fontsize=7.5, color="#555555")

    box(ax, 60, 26, 240, 28, lines=["408 scenes, 2345 x 2351 px"], kind="base")
    box(ax, 60, 74, 240, 28, lines=["1224 image-mask pairs from 408 originals"], kind="base")
    box(ax, 60, 122, 112, 26, lines=["sequence-block", "3 folds"], kind="base", ls=6.2)
    box(ax, 188, 122, 112, 26, lines=["random split", "3 folds"], kind="base", ls=6.2)
    box(ax, 60, 168, 240, 32, title="five architectures, one protocol",
        lines=["30 fold-runs"], kind="added")
    box(ax, 60, 214, 240, 24, lines=["IoU, clDice, producer's and user's accuracy"], kind="base")

    arrow(ax, (180, 54), (180, 74), label="palette LUT, 512 px cache", loff=(76, 0), fs=5.8)
    arrow(ax, (150, 102), (116, 122))
    arrow(ax, (210, 102), (244, 122))
    arrow(ax, (116, 148), (150, 168))
    arrow(ax, (244, 148), (210, 168))
    arrow(ax, (180, 200), (180, 214))

    box(ax, 420, 26, 268, 28, lines=["Stage 1  -  unconditional mask diffusion"], kind="added")
    box(ax, 420, 74, 268, 28, lines=["Stage 2  -  mask-conditioned image synthesis"], kind="added")
    box(ax, 420, 122, 268, 26, lines=["Stage 3  -  chained sampling, 804 pairs"], kind="added")
    box(ax, 420, 168, 268, 32, title="Stage 4  -  R0 804 real, R1 804 real + 804 synthetic",
        lines=["6000 optimizer steps in both arms"], kind="emph")
    box(ax, 420, 214, 268, 24, lines=["frozen fold-1 scorer, layout fidelity"], kind="base")

    for y0, y1 in ((54, 74), (102, 122), (148, 168), (200, 214)):
        arrow(ax, (554, y0), (554, y1))

    arrow(ax, (300, 184), (420, 226), style="dashed", route="vh",
          label="fold-1 checkpoint", loff=(30, -8), fs=5.8)
    note(ax, 374, 246, "no validation scene enters any generator", ha="center")

    finish(fig, "fig04_study_pipeline", "Fig. 4", "Section 4",
           "after the opening paragraph, before Section 4.1", "figure*",
           "The study pipeline. Left: the segmentation benchmark, five architectures trained under one "
           "protocol in two partitioning regimes, thirty fold-runs in total. Right: the generative half "
           "-- unconditional mask diffusion, mask-conditioned image synthesis, chained sampling of "
           "image-label pairs, and the compute-matched downstream experiment. The frozen scorer used "
           "for layout fidelity is the fold-1 segmentation checkpoint, so no validation scene reaches "
           "any stage of the generator.")
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 5 - proposed model architecture  |  figure*, 190 mm  |  end of S4.1; (a) forward pass, (b) training objective
# ------------------------------------------------------------------------------------
def _build_fig05_proposed_model():
    UWa, UHa = 748, 300
    UWb, UHb = 748, 118
    fw = W2
    fha, fhb = panel_height(fw, UWa, UHa), panel_height(fw, UWb, UHb)

    fig, (axa, axb) = plt.subplots(2, 1, figsize=(fw, fha + fhb + 6 * MM), layout="constrained",
                                   gridspec_kw=dict(height_ratios=[UHa, UHb]))
    diagram_axes(axa, UWa, UHa)
    diagram_axes(axb, UWb, UHb)

    # ---------------- panel (a): forward pass
    box(axa, 10, 100, 80, 42, title="input image", lines=["3 x 512 x 512"], kind="base")

    group(axa, 108, 18, 196, 116, "MiT-B0 encoder, unchanged")
    for i, t in enumerate(["stage 1   32 x 128 x 128",
                           "stage 2   64 x 64 x 64",
                           "stage 3   160 x 32 x 32",
                           "stage 4   256 x 16 x 16"]):
        box(axa, 118, 44 + i * 21, 176, 17, lines=[t], kind="base", ls=6.0, round_size=2)

    group(axa, 320, 18, 180, 116, "all-MLP decoder, unchanged")
    box(axa, 328, 46, 164, 20, lines=["project 4 stages to 256 ch, up to H/4"], kind="base",
        ls=6.0, round_size=2)
    box(axa, 328, 70, 164, 18, lines=["concat 1024 x 128 x 128"], kind="base", ls=6.0, round_size=2)
    box(axa, 328, 92, 164, 26, title="1x1 fuse", lines=["FUSED 256 x 128 x 128"], kind="base",
        ts=6.4, ls=6.0, round_size=2)

    box(axa, 520, 40, 160, 42, title="classifier 1x1", lines=["BASE LOGITS 6 x 128 x 128"],
        kind="base")
    note(axa, 600, 90, "vanilla SegFormer stops here", ha="center")

    for i, (t1, t2, t3) in enumerate([("Conv3x3, stride 2", "GN, ReLU", "8 x 256 x 256"),
                                      ("Conv3x3", "GN, ReLU", "16 x 256 x 256"),
                                      ("Conv3x3", "GN, ReLU", "32 x 256 x 256")]):
        box(axa, 108 + i * 112, 166, 104, 46, title=t1, lines=[t2, t3], kind="added",
            ts=6.2, ls=6.0)
    box(axa, 444, 176, 52, 26, lines=["$\\times\\ \\gamma$", "init 0.1"], kind="added", ls=6.2)

    box(axa, 520, 124, 190, 26, lines=["proj 1x1 to 64 ch, up to 64 x 256 x 256"], kind="added",
        ls=6.0)
    box(axa, 520, 160, 190, 22, lines=["concat 96 x 256 x 256"], kind="added", ls=6.0)
    box(axa, 520, 192, 190, 24, lines=["Conv3x3, GN, ReLU, 64 x 256 x 256"], kind="added", ls=6.0)
    box(axa, 520, 226, 190, 30, title="refine 1x1, init $\\sigma$ = 1e-4",
        lines=["RESIDUAL 6 x 256 x 256"], kind="added", ts=6.2, ls=6.0)

    circ(axa, 726, 208, 12, symbol="+", fs=10)
    box(axa, 520, 268, 190, 28, title="final logits 6 x 512 x 512", kind="emph", ts=7.0)

    arrow(axa, (90, 121), (108, 76), route="vh")
    arrow(axa, (304, 76), (320, 76))
    arrow(axa, (500, 61), (520, 61))
    arrow(axa, (50, 142), (108, 189), route="vh", label="raw image", loff=(-14, 6), fs=5.8)
    for i in range(2):
        arrow(axa, (212 + i * 112, 189), (220 + i * 112, 189))
    arrow(axa, (436, 189), (444, 189))          # gate sits after the third conv
    arrow(axa, (496, 189), (520, 171), route="vh")
    arrow(axa, (492, 118), (520, 137), route="vh", label="FUSED", loff=(-2, -8), fs=5.8)
    arrow(axa, (615, 150), (615, 160))
    arrow(axa, (615, 182), (615, 192))
    arrow(axa, (615, 216), (615, 226))
    arrow(axa, (680, 61), (726, 196), route="hv", label="up to 512", loff=(26, -34), fs=5.8)
    arrow(axa, (710, 241), (726, 220), route="vh")
    arrow(axa, (726, 220), (710, 282), route="vh")

    axa.add_patch(Rectangle((10, 288), 9, 7, fc=STYLE["base"]["fc"], ec=STYLE["base"]["ec"], lw=0.8))
    axa.text(23, 292, "SegFormer-B0, untouched", va="center", fontsize=6.2, color="#555555")
    axa.add_patch(Rectangle((150, 288), 9, 7, fc=STYLE["added"]["fc"],
                            ec=STYLE["added"]["ec"], lw=0.8))
    axa.text(163, 292, "added:  3.716 M $\\rightarrow$ 3.794 M parameters, +1.9 %",
             va="center", fontsize=6.2, color="#555555")

    # ---------------- panel (b): training objective
    group(axb, 8, 6, 732, 104, "training only")
    box(axb, 22, 28, 156, 58, title="$w(x)$ reliability weight",
        lines=["from the annotation geometry", "Eq. (1), no learned parameters"],
        kind="added", ts=6.6, ls=6.0)
    box(axb, 250, 22, 244, 24, title="weighted cross-entropy",
        lines=["effective-number weights, label smoothing 0.05"], kind="added", ts=6.4, ls=5.9)
    box(axb, 250, 52, 244, 24, title="Lov\u00e1sz-softmax",
        lines=["$\\lambda$: 0 $\\rightarrow$ 0.75 over ten epochs"], kind="added", ts=6.4, ls=5.9)
    box(axb, 250, 82, 244, 24, title="soft centreline Dice, road and river",
        lines=["$\\mu$: 0 $\\rightarrow$ 0.30 at epoch 20"], kind="added", ts=6.4, ls=5.9)
    box(axb, 536, 46, 58, 34, title="$\\Sigma$", lines=["objective"], kind="emph", ts=9, ls=5.9)
    box(axb, 622, 46, 112, 34, title="presence-balanced",
        lines=["sampler, cap 8"], kind="added", ts=6.4, ls=5.9)

    arrow(axb, (178, 44), (250, 34), route="hv", label="as a weight", loff=(2, -7), fs=5.8)
    arrow(axb, (178, 70), (250, 64), route="hv", style="dashed",
          label="as a mask, $w$ < 0.60", loff=(2, 8), fs=5.8)
    for yy in (34, 64, 94):
        arrow(axb, (494, yy), (536, 63), route="hv")

    finish(fig, "fig05_proposed_model", "Fig. 5", "Section 4.1",
           "at the end of the subsection, after the presence-balanced sampler sentence", "figure*",
           "The proposed model. (a) Forward pass. A SegFormer-B0 backbone (grey) is left untouched: its "
           "four encoder stages are fused by the all-MLP decoder at quarter resolution and classified "
           "there, which is the vanilla prediction. Three components are added (blue): a three-layer "
           "detail path that runs on the raw image at stride 2 and is gated by a learned scalar, "
           "concatenated with the projected decoder feature at half resolution, fused, and reduced to a "
           "six-channel residual added to the upsampled backbone logits. The refine convolution is "
           "initialised at 1e-4 and the gate at 0.1, so at step zero the model reproduces the backbone. "
           "(b) Training objective. The per-pixel reliability weight enters cross-entropy as a weight "
           "and the Lovasz-softmax term as a hard mask, because the latter is a set function over a "
           "sorted error vector rather than a per-pixel sum. The additions cost 0.078 M parameters, "
           "1.9 % of the backbone.",
           axes=[axa, axb], hspace=0.10)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 7 - two-stage generator  |  figure*, 190 mm  |  S4.2, after the Stage-2 paragraph   [S1_PARAMS / S1_STEPS still '??']
# ------------------------------------------------------------------------------------
def _build_fig07_two_stage_generator():
    S1_PARAMS = "?? M"      # <-- 12 M (paper) or 18.5 M (results master section 8.1). Resolve first.
    S1_STEPS  = "??"        # <-- 200 (paper) or 50 (results master Stage-1 table).

    UW, UH = 748, 236
    fw = W2; fh = panel_height(fw, UW, UH)
    fig, ax = plt.subplots(figsize=(fw, fh), layout="constrained")
    diagram_axes(ax, UW, UH)

    box(ax, 8, 78, 76, 30, lines=["$z \\sim \\mathcal{N}(0, I)$"], kind="base", ls=7.0)

    box(ax, 104, 26, 206, 108, title="Stage 1  -  layout",
        lines=["unconditional mask diffusion, " + S1_PARAMS,
               "128 x 128, 7 one-hot channels",
               "v-prediction, zero terminal SNR",
               "deterministic sampling, " + S1_STEPS + " steps"],
        kind="added", ts=7.2, ls=6.0)
    box(ax, 104, 152, 206, 60, title="post-process and upsample",
        lines=["modal-3 / close-0 / thin-30 / bulk-0",
               "nearest neighbour to 512 px"],
        kind="added", ts=6.6, ls=6.0)

    box(ax, 356, 26, 236, 130, title="Stage 2  -  appearance",
        lines=["Stable Diffusion 1.5, frozen",
               "LoRA r16 $\\alpha$16 on to_q, to_k, to_v, to_out.0",
               "control branch from control_v11p_sd15_seg",
               "input conv replaced by a zero-initialised",
               "seven-channel convolution",
               "captions templated from class proportions"],
        kind="added", ts=7.2, ls=6.0)
    note(ax, 474, 168, "only the adapter and the control branch train", ha="center")

    box(ax, 608, 26, 132, 74, title="Stage 3  -  chaining",
        lines=["sample a pool x 2.5", "greedy marginal match"],
        kind="added", ts=7.0, ls=6.0)
    box(ax, 608, 120, 132, 66, title="synthetic pairs",
        lines=["R1  804", "R2  1608", "R3  2412"], kind="emph", ts=7.0, ls=6.2)

    arrow(ax, (84, 93), (104, 80), route="vh")
    arrow(ax, (207, 134), (207, 152))
    arrow(ax, (310, 182), (356, 100), route="vh")
    note(ax, 207, 226, "interface contract: 512 px, class indices", ha="center")
    arrow(ax, (592, 91), (608, 63), route="vh")
    arrow(ax, (674, 100), (674, 120))

    finish(fig, "fig07_two_stage_generator", "Fig. 7", "Section 4.2",
           "after the Stage-2 paragraph, before 'Stage 3, chaining'", "figure*",
           "The two-stage generator. Stage 1 is an unconditional diffusion model over six-class label "
           "maps at 128 x 128, post-processed and upsampled by nearest neighbour to the 512-pixel "
           "interface. Stage 2 paints those masks with a frozen Stable Diffusion 1.5 backbone under "
           "low-rank adaptation and a control branch whose input convolution is replaced by a "
           "zero-initialised seven-channel convolution, so that the condition enters as class channels "
           "rather than as a palette. Stage 3 chains the two and filters greedily from a pool 2.5 times "
           "the target size, so that the class pixel marginal of each set matches the real training "
           "marginal.")
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. S4 - why a binary label code shatters thin classes  |  figure, 90 mm  |  Supplementary, or inset to Fig. 7
# ------------------------------------------------------------------------------------
def _build_figS4_binary_label_code():
    # ---------------- S4: why a binary label code shatters thin classes
    UW, UH = 354, 128
    fw = W1; fh = panel_height(fw, UW, UH)
    fig, ax = plt.subplots(figsize=(fw, fh), layout="constrained")
    diagram_axes(ax, UW, UH)

    ax.plot([40, 314], [34, 34], color="#8C8C8C", lw=1.0)
    for x, lab, code_, kind in ((40, "residential", "000", "base"),
                                (177, "river", "011", "emph"),
                                (314, "forest", "111", "base")):
        ax.plot([x], [34], marker="o", ms=5, color=STYLE[kind]["ec"], zorder=3)
        ax.text(x, 22, lab, ha="center", fontsize=6.6, color=TXT)
        ax.text(x, 46, code_, ha="center", fontsize=6.6, color="#666666", family="monospace")
    ax.annotate("", xy=(40, 60), xytext=(314, 60),
                arrowprops=dict(arrowstyle="|-|", lw=0.8, color="#B2182B",
                                shrinkA=0, shrinkB=0, mutation_scale=3))
    ax.text(177, 70, "the most common boundary in the dataset", ha="center", fontsize=6.2,
            color="#B2182B")
    ax.text(177, 82, "and its midpoint in code space is river", ha="center", fontsize=6.2,
            color="#B2182B")
    box(ax, 30, 96, 294, 24,
        lines=["thresholding channels independently writes river wherever one bit crosses first"],
        kind="base", ls=6.0)
    ax.text(177, 8, "binary label code, one bit per axis", ha="center", fontsize=7, color="#555555")

    finish(fig, "figS4_binary_label_code", "Fig. S4", "Supplementary",
           "or Section 4.2 as an inset to Fig. 7", "figure",
           "Why a binary label code fails on thin classes. Under the three-channel code inherited from "
           "the antecedent work, river is the exact midpoint between residential and forest, which is "
           "the most common class boundary in the corpus. A single bit crossing first therefore writes "
           "river along that boundary, and structures two to three pixels wide at the training "
           "resolution are shattered: road and river arrive 6.6 and 6.1 times more fragmented than in "
           "real masks. One-hot coding gives every class an orthogonal channel and removes the "
           "midpoint.")
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. S5 - the three causes of rare-class failure  |  figure*, 190 mm  |  S7.1, or Supplementary
# ------------------------------------------------------------------------------------
def _build_figS5_failure_decomposition():
    # ---------------- S5: the three separate causes of rare-class failure
    UW, UH = 748, 176
    fw = W2; fh = panel_height(fw, UW, UH)
    fig, ax = plt.subplots(figsize=(fw, fh), layout="constrained")
    diagram_axes(ax, UW, UH)

    COLS = [("UnusedLand", "#C2A477",
             ["spectral inseparability",
              "JM = 0.051 against agricultural",
              "architecture-invariant:\nblock-mean spread 0.053, the\ntightest of any class",
              "a near-infrared band"]),
            ("Road", "#4D4D4D",
             ["geometric under-resolution",
              "2.10 px wide at the H/4 stride",
              "over-paints: user's accuracy\n0.52 against producer's 0.73",
              "a higher-resolution prediction path"]),
            ("River", "#1F6FB4",
             ["statistical rarity and clustering",
              "83 tiles, spatially concentrated",
              "largest architecture spread of\nany class, a factor of 3.8",
              "more examples, or generation"])]
    ROWS = ["cause", "evidence", "diagnostic signature", "what could fix it"]
    RY = [40, 72, 100, 148]
    RH = [26, 22, 42, 22]

    for r, (lab, y, h) in enumerate(zip(ROWS, RY, RH)):
        ax.text(96, y + h / 2, lab, ha="right", va="center", fontsize=6.6,
                color="#666666", style="italic")

    for c, (name, col, vals) in enumerate(COLS):
        x = 108 + c * 214
        ax.add_patch(FancyBboxPatch((x, 8), 200, 22, boxstyle="round,pad=0,rounding_size=3",
                                    fc=col, ec="none", zorder=2))
        ax.text(x + 100, 19, name, ha="center", va="center", fontsize=7.5,
                color=("white" if name != "UnusedLand" else "#2A2A2A"), zorder=3)
        for r, (v, y, h) in enumerate(zip(vals, RY, RH)):
            box(ax, x, y, 200, h, lines=v.split("\n"),
                kind=("added" if r == 3 else "base"), ls=6.1, round_size=2)

    finish(fig, "figS5_failure_decomposition", "Fig. S5", "Section 7.1",
           "after the paragraph decomposing the three weak classes, or Supplementary", "figure*",
           "Three separate causes of rare-class failure, with the measurement that identifies each and "
           "the intervention it implies. Treating the three weak classes as one problem is the natural "
           "move and the measurements say it is wrong: only river's failure mode is one a generative "
           "intervention could plausibly address.")
    plt.show()


print('defined: 5 figure builders')

In [ ]:
# ====================================================================================
#  GROUP 3  ·  DATASET PHOTOS
#  Original code, with the three marked edits: drawn locator, row labels from the selection, and the layout-mIoU inset.
# ====================================================================================


# ------------------------------------------------------------------------------------
# Fig. 1 - study area and sample tiles  |  figure*, 190 mm  |  end of S3.1
# ------------------------------------------------------------------------------------
def _build_fig01_study_area_and_tiles():
    INCLUDE_LOCATOR = True     # False if you would rather paste the QGIS map in separately

    stems = FIG1_TILES if FIG1_TILES else [stem(p) for p in pick(REAL_IMGS, 3)]
    if DEMO:
        stems = ["demo1", "demo2", "demo3"]
    rows = []
    for i, s in enumerate(stems if stems else ["", "", ""]):
        if DEMO:
            im, mk = demo_tile(10 + i), demo_tile(10 + i, "mask")
        else:
            im = load(by_stem(REAL_IMGS, s)) if by_stem(REAL_IMGS, s) else None
            mk = recolour(load(by_stem(REAL_MASKS, s))) if by_stem(REAL_MASKS, s) else None
        rows.append((s, im, mk))

    ncol = 3 + (1 if INCLUDE_LOCATOR else 0)
    fig, axs = plt.subplots(3, ncol, figsize=(W2, W2 * 3 / ncol * 1.02), layout="constrained")
    axs = np.atleast_2d(axs)

    if INCLUDE_LOCATOR:
        gs = axs[0, 0].get_gridspec()
        for a in axs[:, 0]:
            a.remove()
        axloc = fig.add_subplot(gs[:, 0])
        axloc.set_xticks([]); axloc.set_yticks([])
        for sp in axloc.spines.values():
            sp.set_linewidth(0.5); sp.set_color("0.6")
        draw_locator(axloc)
        col0 = 1
    else:
        col0 = 0

    COLTITLES = ["image", "annotation", "overlay"]
    for r, (s, im, mk) in enumerate(rows):
        for c, arr in enumerate([im, mk, overlay(im, mk)]):
            ax = axs[r, col0 + c]
            show(ax, arr, slot="FIG1_TILES[" + str(r) + "]", demo=DEMO)
            if r == 0:
                ax.set_title(COLTITLES[c], fontsize=7, pad=2)
        if col0 == 0:
            axs[r, 0].set_ylabel(FIG1_NOTES[r], fontsize=6.5)
        else:
            axs[r, col0].set_ylabel(FIG1_NOTES[r], fontsize=6.5)

    class_legend(fig)
    tile_axes = [axs[r, c] for r in range(3) for c in range(col0, ncol)]
    panels = ([axloc, tile_axes] if INCLUDE_LOCATOR else [tile_axes])

    finish(fig, "fig01_study_area_and_tiles", "Fig. 1", "Section 3.1",
           "at the end of the subsection, after the sentence on the plus or minus 1.6 % scale "
           "uncertainty", "figure*",
           "The corpus. (a) Approximate extent of the 202.4 km2 annotated by 408 contiguous tiles "
           "within the Kathmandu Valley, Bagmati Province, Nepal. Per-tile footprints were not "
           "preserved, so no georeferenced tile index is shown. Map lines delineate study areas and do "
           "not necessarily depict accepted national boundaries. (b-d) Three tiles as image, annotation "
           "and overlay, spanning the dense historic core, a river corridor and the cultivated fringe. "
           "The palette shown is used in every subsequent figure.",
           axes=panels)
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 6 - the reliability weight field  |  figure*, 190 mm  |  S4.1, immediately after Eq. (1)
# ------------------------------------------------------------------------------------
def _build_fig06_reliability_weight():
    from scipy import ndimage as ndi

    TAU_LINEAR, TAU_AREA = 0.35, 0.60
    W_MIN = 0.25
    D_MAX_M = 14.07                     # metric cap, from the code that actually ran
    M_PER_PX = 703.5 / 512.0            # working resolution

    def mask_to_labels(mask_rgb):
        lab = np.full(mask_rgb.shape[:2], 255, np.uint8)
        for i, c in enumerate(CLASSES):
            lab[np.all(mask_rgb == hex2rgb(CLR[c]), -1)] = i
        return lab

    def csa_weight(lab):
        # boundary set B: a pixel whose 4-neighbourhood carries a different label
        B = np.zeros(lab.shape, bool)
        B[:-1, :] |= lab[:-1, :] != lab[1:, :]
        B[1:, :]  |= lab[1:, :]  != lab[:-1, :]
        B[:, :-1] |= lab[:, :-1] != lab[:, 1:]
        B[:, 1:]  |= lab[:, 1:]  != lab[:, :-1]
        B[0, :] = B[-1, :] = B[:, 0] = B[:, -1] = False        # exclude the image border
        D = ndi.distance_transform_edt(~B)
        d_max_px = D_MAX_M / M_PER_PX
        b = np.ones_like(D)
        for c in range(6):
            tau = TAU_LINEAR if c in (4, 5) else TAU_AREA      # road and river are the linear classes
            comp, n = ndi.label(lab == c)
            if n == 0:
                continue
            means = ndi.mean(D, comp, index=np.arange(1, n + 1))
            for k in range(1, n + 1):
                R = 2.0 * means[k - 1]
                b[comp == k] = np.clip(tau * R, 1.0, d_max_px)
        w = np.clip(D / np.maximum(b, 1e-6), W_MIN, 1.0)
        return B, D, w

    s6 = FIG6_TILE if FIG6_TILE else (stem(REAL_IMGS[0]) if REAL_IMGS else "")
    if DEMO:
        img6, msk6 = demo_tile(77), demo_tile(77, "mask")
    else:
        img6 = load(by_stem(REAL_IMGS, s6)) if by_stem(REAL_IMGS, s6) else None
        msk6 = recolour(load(by_stem(REAL_MASKS, s6))) if by_stem(REAL_MASKS, s6) else None
    img6, msk6 = crop(img6, FIG6_CROP), crop(msk6, FIG6_CROP)

    fig, axs = plt.subplots(1, 4, figsize=(W2, W2 / 4 * 1.16), layout="constrained")
    if msk6 is None:
        for i, ax in enumerate(axs):
            show(ax, None, slot="FIG6_TILE")
        Bv = Dv = wv = None
    else:
        Bv, Dv, wv = csa_weight(mask_to_labels(msk6))
        ann = msk6.copy(); ann[Bv] = 255
        show(axs[0], img6, demo=DEMO)
        show(axs[1], ann, demo=DEMO)
        axs[2].imshow(Dv, cmap="magma"); axs[2].set_xticks([]); axs[2].set_yticks([])
        im3 = axs[3].imshow(wv, cmap="viridis", vmin=W_MIN, vmax=1.0)
        axs[3].set_xticks([]); axs[3].set_yticks([])
        for ax in axs[2:]:
            for sp in ax.spines.values():
                sp.set_linewidth(0.5); sp.set_color("0.6")
        cb = fig.colorbar(im3, ax=axs[3], fraction=0.046, pad=0.02)
        cb.set_label("$w(x)$", fontsize=7); cb.ax.tick_params(labelsize=6)
        cb.outline.set_linewidth(0.5)

    finish(fig, "fig06_reliability_weight", "Fig. 6", "Section 4.1",
           "immediately after Eq. (1) and the sentence giving its parameters", "figure*",
           "The per-pixel reliability weight of Eq. (1), computed on one crop containing both a road and "
           "a large residential block. (a) Image. (b) Annotation with the boundary set B overdrawn in "
           "white. (c) Euclidean distance transform of the complement of B. (d) The resulting weight "
           "field, clipped to a floor of 0.25. Because the unreliable band is expressed as a fraction of "
           "local object width rather than a fixed number of pixels, it stays narrow on the road and "
           "widens on the residential polygon; a fixed-pixel band would drive the two linear classes to "
           "the floor and suppress exactly the classes the mechanism exists to rescue.",
           axes=list(axs))
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 9 - qualitative comparison  |  figure*, 190 mm  |  end of S6.2
# ------------------------------------------------------------------------------------
def _build_fig09_qualitative_comparison():
    ROWLAB = FIG9_ROWLAB      # set by the selection / inference cells
    COLLAB = ["image", "reference annotation", "SegFormer-B0", "proposed model"]

    s9 = FIG9_SCENES if FIG9_SCENES else [stem(p) for p in pick(REAL_IMGS, 3)]
    if DEMO:
        s9 = ["d1", "d2", "d3"]

    fig, axs = plt.subplots(3, 4, figsize=(W2, W2 * 3 / 4 * 1.03), layout="constrained")
    for r in range(3):
        s = s9[r] if r < len(s9) else ""
        if DEMO:
            im, gt = demo_tile(40 + r), demo_tile(40 + r, "mask")
            b0, tc = demo_tile(140 + r, "mask"), demo_tile(240 + r, "mask")
        else:
            im = load(by_stem(REAL_IMGS, s)) if by_stem(REAL_IMGS, s) else None
            gt = recolour(load(by_stem(REAL_MASKS, s))) if by_stem(REAL_MASKS, s) else None
            pb = by_stem(PREDS, FIG9_PRED_B0[r]) if r < len(FIG9_PRED_B0) else None
            pt = by_stem(PREDS, FIG9_PRED_TC[r]) if r < len(FIG9_PRED_TC) else None
            b0 = recolour(load(pb)) if pb else None
            tc = recolour(load(pt)) if pt else None
        slots = ["FIG9_SCENES[" + str(r) + "]"] * 2 + ["FIG9_PRED_B0[" + str(r) + "]",
                                                       "FIG9_PRED_TC[" + str(r) + "]"]
        for c, arr in enumerate([im, gt, b0, tc]):
            show(axs[r, c], arr, slot=slots[c], demo=DEMO)
            if r == 0:
                axs[r, c].set_title(COLLAB[c], fontsize=7, pad=2)
        axs[r, 0].set_ylabel(ROWLAB[r], fontsize=6.5)

    class_legend(fig)

    finish(fig, "fig09_qualitative_comparison", "Fig. 9", "Section 6.2",
           "at the end of the subsection, after the sentence on the dominant error modes", "figure*",
           "Qualitative comparison on held-out scenes of block fold 1. Rows are the highest- and "
           "lowest-scoring validation scenes and one intermediate case. The dominant error modes are "
           "the ones the separability matrix predicts: vacant ground absorbed into cultivated ground on "
           "the valley fringe, urban tree cover claimed as forest, and river fragmenting where it "
           "narrows.")
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. 11 - synthetic pairs and read-back  |  figure*, 190 mm  |  end of S6.4
# ------------------------------------------------------------------------------------
def _build_fig11_synthetic_pairs():
    ROWLAB11 = FIG11_ROWLAB   # set by the selection / inference cells
    COLLAB11 = ["generated mask", "generated image", "frozen scorer read-back"]

    p11 = FIG11_PAIRS if FIG11_PAIRS else [stem(p) for p in pick(FAKE_MASKS, 3)]
    if DEMO or not FAKE_MASKS:
        p11 = ["f1", "f2", "f3"]

    fig, axs = plt.subplots(3, 3, figsize=(W2, W2 * 3 / 3 * 0.80), layout="constrained")
    for r in range(3):
        s = p11[r] if r < len(p11) else ""
        if DEMO or not FAKE_MASKS:
            gm, gi, rb = demo_tile(60 + r, "mask"), demo_tile(60 + r), None
        else:
            gm = recolour(load(by_stem(FAKE_MASKS, s))) if by_stem(FAKE_MASKS, s) else None
            gi = load(by_stem(FAKE_IMGS, s)) if by_stem(FAKE_IMGS, s) else None
            pr = by_stem(PREDS, FIG11_READBACK[r]) if r < len(FIG11_READBACK) else None
            rb = recolour(load(pr)) if pr else None
        slots = ["FIG11_PAIRS[" + str(r) + "]"] * 2 + ["FIG11_READBACK[" + str(r) + "]"]
        for c, arr in enumerate([gm, gi, rb]):
            show(axs[r, c], arr, slot=slots[c], demo=(DEMO or not FAKE_MASKS))
            if r == 0:
                axs[r, c].set_title(COLLAB11[c], fontsize=7, pad=2)
        if FIG11_SCORES[r] is not None and rb is not None:
            axs[r, 2].text(0.97, 0.04, "layout mIoU %.3f" % FIG11_SCORES[r],
                           transform=axs[r, 2].transAxes, ha="right", va="bottom",
                           fontsize=6.0, color="white",
                           bbox=dict(fc="0.15", ec="none", pad=1.2, alpha=0.75))
        axs[r, 0].set_ylabel(ROWLAB11[r], fontsize=6.5)

    class_legend(fig)

    finish(fig, "fig11_synthetic_pairs", "Fig. 11", "Section 6.4",
           "at the end of the subsection, after the presence-rate sentence", "figure*",
           "Three synthetic pairs spanning the fidelity range: the generated mask, the image painted "
           "from it, and the frozen scorer's read-back of that image against the mask that generated it. "
           "The bottom row shows the failure that drives the downstream result -- river arriving as a "
           "wide fragmented blob rather than a narrow continuous ribbon, which teaches a false shape "
           "prior on top of a false texture.")
    plt.show()


# ------------------------------------------------------------------------------------
# Fig. A.1(a) - the pitch crop  |  figure, 90 mm  |  Appendix A, beside the budget panel
# ------------------------------------------------------------------------------------
def _build_figA1a_pitch_crop():
    # fractions of the crop, (x0, y0, x1, y1) in axes coordinates - adjust once you see the image
    BASELINES = [(0.20, 0.28, 0.20, 0.72),     # penalty area width
                 (0.20, 0.28, 0.36, 0.28),     # penalty area depth
                 (0.28, 0.40, 0.28, 0.60),     # goal area width
                 (0.28, 0.40, 0.34, 0.40)]     # goal area depth
    BLAB = ["40.32 m", "16.50 m", "18.32 m", "5.50 m"]

    sA = FIGA1_TILE if FIGA1_TILE else (stem(REAL_IMGS[0]) if REAL_IMGS else "")
    imgA = demo_tile(5) if DEMO else (load(by_stem(REAL_IMGS, sA)) if by_stem(REAL_IMGS, sA) else None)
    imgA = crop(imgA, FIGA1_CROP)

    fig, ax = plt.subplots(figsize=(W1, W1 * 0.72), layout="constrained")
    show(ax, imgA, slot="FIGA1_TILE", demo=DEMO)
    if imgA is not None:
        for (x0, y0, x1, y1), lab in zip(BASELINES, BLAB):
            ax.plot([x0, x1], [y0, y1], transform=ax.transAxes, color="#B2182B", lw=1.0,
                    marker="|", ms=4, mew=1.0)
            ax.text((x0 + x1) / 2 + 0.02, (y0 + y1) / 2, lab, transform=ax.transAxes,
                    fontsize=6.0, color="#B2182B", va="center",
                    bbox=dict(fc="white", ec="none", pad=0.8, alpha=0.85))

    finish(fig, "figA1a_pitch_crop", "Fig. A.1(a)", "Appendix A",
           "beside the error-budget panel from the first notebook", "figure",
           "The football pitch after automatic location and rotation to upright, with the four "
           "regulation-fixed controls marked. Penalty-area and goal-area dimensions are fixed by the "
           "Laws of the Game independently of the size of the pitch on which they are marked, which is "
           "what makes them usable as a scale reference.")
    plt.show()


# ------------------------------------------------------------------------------------
# Graphical abstract  |  133 x 53 mm, >= 531 x 1328 px  |  separate upload, not in the manuscript body
# ------------------------------------------------------------------------------------
def _build_graphical_abstract():
    GA_W, GA_H = 1328 / 100, 531 / 100      # inches at 100 px per inch, then saved at 600 dpi
    fig, axs = plt.subplots(1, 3, figsize=(GA_W, GA_H), layout="constrained",
                            gridspec_kw=dict(width_ratios=[1, 1, 1]))

    # panel 1 - the corpus
    ax = axs[0]
    sub = fig.add_subplot(1, 3, 1); sub.remove()
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    ax.text(0.5, 0.93, "A CHARACTERISED CORPUS", ha="center", fontsize=12, transform=ax.transAxes)
    ax.text(0.5, 0.06, "202.4 km$^2$  ·  408 scenes  ·  six classes\n"
            "every class pair inseparable in RGB", ha="center", fontsize=9,
            transform=ax.transAxes)
    t_im = demo_tile(1) if DEMO or not REAL_IMGS else load(REAL_IMGS[0])
    t_mk = demo_tile(1, "mask") if DEMO or not REAL_MASKS else recolour(load(REAL_MASKS[0]))
    for k, (arr, x0) in enumerate([(t_im, 0.04), (t_mk, 0.52)]):
        sax = ax.inset_axes([x0, 0.26, 0.44, 0.58])
        show(sax, arr, demo=DEMO)

    # panel 2 - the partition result
    ax = axs[1]
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    ax.text(0.5, 0.93, "THE PARTITION MATTERS MORE\nTHAN THE ARCHITECTURE", ha="center",
            fontsize=12, transform=ax.transAxes)
    inner = ax.inset_axes([0.14, 0.24, 0.78, 0.48])
    archs = ["U-Net", "UNetFormer", "SegFormer-B0", "SegNeXt-T", "proposed"]
    blk = [0.3986, 0.4200, 0.3962, 0.4723, 0.4682]
    rnd = [0.4494, 0.5058, 0.4765, 0.5531, 0.5631]
    y = np.arange(5)[::-1]
    for k in range(5):
        inner.plot([blk[k], rnd[k]], [y[k], y[k]], color="0.75", lw=1.2)
        inner.plot(blk[k], y[k], "o", color="#B2182B", ms=5)
        inner.plot(rnd[k], y[k], "s", color="#2166AC", ms=4.5)
    inner.set_yticks(y); inner.set_yticklabels(archs, fontsize=8)
    inner.set_xlim(0.36, 0.60); inner.tick_params(labelsize=7)
    inner.set_xlabel("mean IoU", fontsize=8)
    inner.spines["top"].set_visible(False); inner.spines["right"].set_visible(False)
    ax.text(0.5, 0.06, "0.051 to 0.095 mIoU, every architecture,\nfor the split protocol alone",
            ha="center", fontsize=9, transform=ax.transAxes)

    # panel 3 - the generative result
    ax = axs[2]
    ax.set_xticks([]); ax.set_yticks([]); ax.axis("off")
    ax.text(0.5, 0.93, "SYNTHETIC PAIRS DO NOT HELP", ha="center", fontsize=12,
            transform=ax.transAxes)
    inner = ax.inset_axes([0.16, 0.30, 0.74, 0.42])
    inner.barh([1, 0], [-0.0140, -0.0570], color=["#8C8C8C", "#1F6FB4"], height=0.5)
    inner.set_yticks([1, 0]); inner.set_yticklabels(["mean IoU", "river IoU"], fontsize=8)
    inner.axvline(0, color="0.2", lw=0.8)
    inner.set_xlim(-0.075, 0.012); inner.tick_params(labelsize=7)
    inner.set_xlabel("change from adding synthetic pairs", fontsize=8)
    inner.spines["top"].set_visible(False); inner.spines["right"].set_visible(False)
    ax.text(0.5, 0.06, "all five architectures lose, and the damage\n"
            "lands on the rare thin classes it was built to enrich",
            ha="center", fontsize=9, transform=ax.transAxes)

    fig.savefig(OUT / "graphical_abstract.pdf")
    fig.savefig(OUT / "graphical_abstract.png", dpi=600)
    REGISTER.append(dict(file="graphical_abstract.pdf", figure="Graphical abstract",
                         section="separate upload", insertion_point="not in the manuscript body",
                         float_env="-", size="133 x 53 mm",
                         caption="No caption. The graphical abstract stands alone."))
    print("graphical_abstract.pdf   1328 x 531 px minimum satisfied")
    plt.show()


print('defined: 6 figure builders')

## 12 · Selector, register and zip

In [ ]:
import re

import csv, shutil, os, traceback

# ---------------------------------------------------------------------------------
#  The three groups. Order inside each group is the order the figures are built in.
# ---------------------------------------------------------------------------------
GROUPS = {
    1: ("plotted figures (data-driven)", [
        "fig02_corpus_difficulty", "fig03_partitioning_regimes", "fig08_partition_effect",
        "fig14_proposed_vs_backbone", "fig15_perclass_iou_penalty", "fig16_loveda_perclass",
        "fig10_generator_selection", "fig20_synthetic_pool_quality",
        "fig12_downstream_forest_plot", "fig17_perclass_downstream_delta",
        "fig18_mechanism_fidelity_ratio", "fig13_component_ablation",
        "fig21_fwiou_and_cldice", "fig19_label_representation",
        "figA1_scale_recovery_budget", "figS1_efficiency"]),
    2: ("block diagrams (drawn)", [
        "fig04_study_pipeline", "fig05_proposed_model", "fig07_two_stage_generator",
        "figS4_binary_label_code", "figS5_failure_decomposition"]),
    3: ("photos from the dataset (montages)", [
        "fig01_study_area_and_tiles", "fig06_reliability_weight",
        "fig09_qualitative_comparison", "fig11_synthetic_pairs",
        "figA1a_pitch_crop", "graphical_abstract"]),
}

ALL_NAMES = [n for g in (1, 2, 3) for n in GROUPS[g][1]]
COLS = ["file", "figure", "section", "insertion_point", "float_env", "size", "caption"]


def _select(choice):
    """Accepts 1 | '1' | '13' | '1,3' | '1 3' | 'all' | [1, 3] | None."""
    if choice is None or choice == "":
        return [1, 2, 3]
    if isinstance(choice, (list, tuple, set)):
        picked = [int(c) for c in choice]
    else:
        s = str(choice).strip().lower()
        if s in ("all", "a", "*", "123", "everything"):
            return [1, 2, 3]
        picked = [int(d) for d in re.findall(r"[123]", s)]
    picked = sorted(set(picked))
    if not picked:
        raise ValueError("choose 1, 2, 3, any combination such as '1,3', or 'all'")
    return picked


def _write_register():
    rows = {}
    for r in REGISTER:                       # last write of a file wins
        rows[r["file"]] = r
    order = [n + ".pdf" for n in ALL_NAMES]
    keep = [rows[f] for f in order if f in rows and (OUT / f).exists()]
    keep += [r for f, r in rows.items() if f not in order and (OUT / f).exists()]
    path = OUT / "figure_register.csv"
    with open(path, "w", newline="", encoding="utf-8-sig") as fh:
        w = csv.DictWriter(fh, fieldnames=COLS); w.writeheader()
        for r in keep:
            w.writerow(r)
    return path, keep


def _zip_figs():
    dest = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
    zp = shutil.make_archive(str(dest / "geonusaf_figures_all"), "zip", root_dir=OUT)
    print("\nzip ->", zp, "(" + str(round(os.path.getsize(zp) / 1e6, 2)) + " MB)")
    try:
        from IPython.display import FileLink, display
        display(FileLink(os.path.relpath(zp, ".")))
    except Exception:
        pass
    print("On Kaggle: Save Version, then take the zip from the Output tab. In an interactive")
    print("session the link above downloads it directly.")
    return zp


def run(choice="all", zip_output=True):
    """1 = plots, 2 = block diagrams, 3 = dataset photos. run('1'), run('1,3'), run('all')."""
    picked = _select(choice)
    print("=" * 86)
    print("BUILDING:", ", ".join(str(g) + " - " + GROUPS[g][0] for g in picked))
    print("=" * 86, "\n")

    made, failed = [], []
    for g in picked:
        title, names = GROUPS[g]
        print("\n" + "-" * 86)
        print("GROUP " + str(g) + " - " + title + "  (" + str(len(names)) + " figures)")
        print("-" * 86)
        for nm in names:
            fn = globals().get("_build_" + nm)
            if fn is None:
                print("  MISSING BUILDER:", nm); failed.append(nm); continue
            try:
                fn()
                made.append(nm)
            except Exception:
                failed.append(nm)
                print("\n*** FAILED: " + nm + " ***")
                traceback.print_exc()
                print("*** carrying on with the rest ***\n")

    path, keep = _write_register()
    print("\n" + "=" * 86)
    print("%-36s %-20s %-9s %s" % ("FILE", "FIGURE", "FLOAT", "SECTION"))
    print("-" * 86)
    for r in keep:
        print("%-36s %-20s %-9s %s" % (r["file"], r["figure"], r["float_env"], r["section"]))
    print("-" * 86)
    print(len(made), "built this run ·", len(keep), "figures now in", OUT.resolve())
    print("register ->", path.resolve())
    if failed:
        print("FAILED:", ", ".join(failed))
    if zip_output:
        _zip_figs()
    return made


def build(*names, zip_output=False):
    """Rebuild single figures by file name: build('fig12_downstream_forest_plot')."""
    for nm in names:
        fn = globals().get("_build_" + nm)
        if fn is None:
            print("no such figure:", nm, "\n  choose from:", ", ".join(ALL_NAMES)); continue
        fn()
    _write_register()
    if zip_output:
        _zip_figs()


def clean():
    """Empty figs/ - only if you want a zip containing this run and nothing else."""
    for p in OUT.glob("*"):
        p.unlink()
    REGISTER.clear()
    print("figs/ emptied")


print("selector ready.")
print("   run(1)      plotted figures      (" + str(len(GROUPS[1][1])) + ")")
print("   run(2)      block diagrams       (" + str(len(GROUPS[2][1])) + ")")
print("   run(3)      dataset photos       (" + str(len(GROUPS[3][1])) + ")")
print("   run('1,3')  any combination      ·   run('all')  everything (" + str(len(ALL_NAMES)) + ")")

---
## 13 · BUILD

In [ ]:
# =====================================================================================
#   BUILD.  CHOICE was set in the control panel at the top of the notebook.
#   You can also call run("2") or build("fig12_downstream_forest_plot") directly.
# =====================================================================================

run(CHOICE)

In [ ]:
# Optional: real clickable buttons, if ipywidgets is available (it is on Kaggle).
# Typing run("1") in the cell above does exactly the same thing.
try:
    import ipywidgets as W
    from IPython.display import display

    _out = W.Output()

    def _mk(label, choice):
        b = W.Button(description=label, layout=W.Layout(width="230px"))
        def _cb(_):
            _out.clear_output()
            with _out:
                run(choice)
        b.on_click(_cb)
        return b

    display(W.VBox([
        W.HTML("<b>GeoNUSAF figures</b> &nbsp; pick a group:"),
        W.HBox([_mk("1 · plotted figures", "1"),
                _mk("2 · block diagrams", "2"),
                _mk("3 · dataset photos", "3"),
                _mk("all three", "all")]),
        _out]))
except Exception as _e:
    print("ipywidgets not available -", _e)
    print('Use the cell above instead:  CHOICE = "1"   then run the cell.')

---
## 14 · Notes

**Verify a PDF before submitting**

```bash
pdfinfo  figs/fig12_downstream_forest_plot.pdf   # 539 pt = 190 mm · 397 = 140 · 255 = 90
pdffonts figs/fig12_downstream_forest_plot.pdf   # serif embedded, no Type 3
```

Insert with `\includegraphics[width=\linewidth]` and no scale factor.

**If Fig. 9 or Fig. 11 still has an empty prediction column**

The prediction panels need model output, and model output needs either

- `PRED_DIR` pointed at prediction PNGs you exported from your own training notebook, or
- internet on in this Kaggle session, so section 9 can pull the checkpoints.

The B0 checkpoint carries its own `model_config`, so it rebuilds itself. TC-SegFormer
is a custom module, and its checkpoint is a bare `model_state`: the notebook rebuilds
the architecture from Section 4.1 and refuses to load unless the parameter names and
shapes match your file exactly. If it refuses, it prints the top-level keys it found —
assign your own class to `BUILD_TC_MODEL`, or export the predictions.

Nothing in this notebook invents model output. An empty panel means the output was not
available, which is a state you can fix; a plausible-looking wrong panel is not.

**Still open, unchanged from the source notebooks**

- `S1_PARAMS` / `S1_STEPS` in Fig. 7 are still `??` — 12 M (paper) vs 18.5 M (results
  master §8.1), and 200 deterministic steps (paper) vs DDIM 50.
- The Stage-2 denoising trace has only its two endpoints, so Fig. 10 draws that segment
  dashed and says so.
- Fig. 9's two prediction columns must come from one code path; the part-1 qualitative
  PNGs in the model repos carry the BGR-as-RGB bug and must not be pasted in.
- The data-availability statement still promises a reconstruction that §3.1 says is
  impossible. That one is a reject risk and no figure can fix it.

**Housekeeping**

- `run()` rezips the whole of `figs/`. `clean()` empties it first if you want a zip of
  one group only.
- The census is cached in `figs/_cache/tile_census.csv`; delete it or call
  `build_census(force=True)` after changing the dataset.
